<a href="https://colab.research.google.com/github/david-levin11/Verification_Notebooks/blob/main/NBM_Percentile_in_Context_Enhanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NBM Percentile in Context**
A script to grab NBM percentile data, deterministic forecast and obs, and put the obs or deterministic forecast into the context of Probabilistic NBM space. This is based on code originally developed by Caleb Steele of Western Region.

-*Steve Levine - NWS MDL/Statistical Modeling Division - 2-March-2023*<br/>
-*David Levin - NWS Alaska Region ESSD - 5-July-2024*
<br/> <br/>
Updates:
> ***BUG:  StageIV QPE data no longer works due to new NWPS...need to reference IDP GIS***

> 15-Jul-2024 Fixed bug which was not displaying map for all Alaska Region

>15-Jul-2024 Included functionality to look at observations within a custom bounding box

> 5-Jul-2024 Included OCONUS functionality (AR and HFO and Puerto Rico) and also the ability to use the 24hr APRFC QC'ed gauge data as a verification source for PQPF 24hr products

> 20-Sept-2023: Included ability to work with HI/PR/AK domains (still no GU yet)

>14-Sep-2023: Updated snow plotting to work with 'unknown' name in snow total grib2 message

>10-Jul-2023: Corrected smoothling spline of percentiles so that now spline passes through relevant end points (set s = 0)

> 20-June 2023: Added deterministic 24-hour snow forecasts, which include same caveats as 24-hour QPF forecasts (random bin assignment, etc.)

> 4-Apr-2023: Added probabilistic (but not determinisitc) 24-hour snow forecasts based on NOHRSC analysis, added zero-padding for time strings where needed.  Set observed or deterministic qpf/snow to zero when value is < 0.005 inches.  Also added minor edits to plots (CWA's are now outlined in black).

> 30-Mar-2023: Fixed bug where 0.0 qpf obs were being set to NaN.  Also, assigned such obs to a random percentile bin where percentile values was also 0.0.

> 13-Mar-2023: Fixed bug with random extra obs showing up.  Also inserted exception for maximum winds, which do not exist in core/deterministic.  Fatal error will generate when attempting to work with core/deterministic max wind.

> 2-Mar-2023: Added capability for maximum wind forecast.  Note that a determinsitic/core maximum wind forecast is not available.

>>Previous development was from Caleb Steele

> 12-Aug-2022: Pretty significant update that changed much of the underlying code. Files used are now grib files (instead of the geotiffs used before), which are larger and take longer to download and decode. The gribs include more percentiles, and a cubic spline is utilized vs linear interpolation now, so the NBM distribution is much better sampled. While all 99 percentiles are avaiable, trimmed it to use 13 (1st, 5th, 10th, 20th, 30th, etc.) to save some time. If you are more patient, you can dig into the code and swap out the lists (uncomment one, uncomment the other) that will use all 99, but it will take awhile. In my limited tests, it offers little improvement in the representation of the CDF/PDF. Also wrapped everything up behind a form and added light/dark mode option. Finally, added an option to generate a csv from the dataframe that is created (helpful to make sure it has done what you expect).

> 7-Apr-2022: Added QPF, but still a lot of cleanup required. If you select Deterministic and QPF, it will really use the percentile mean. Will clean it up to use the deterministic, and add percentile mean as a separate option at some point.

> 18-Feb-2022: Added option of adding CWA boundaries, cleaned up the directories (by actually making some), and more plot tweaks so they all look as expected.

> 17-Feb-2022: fixed a hard coded reference that lead to the histogram always displaying the observation percentile distribution, even when deterministic was selected in regional plots.

> 16-Feb-2022: added "compare_to" variable which lets you switch between comparing obs and NBM determinsitic to the ProbMaxT Percentiles.

> 14-Jul-2025:  Added the ability to work with NBM 5.0 data and added MaxWind for Alaska

> 2-Jun-2026:  Updated notebook so NBM v5.0 is treated as the default/only operational workflow; removed old NBM v4.3/experimental version selection logic.

> 26-Aug-2025:  Updated import procedure to remove dependency on conda and to run with the latest update to colab to python 3.12
> 2-Jun-2026:  Added run-summary diagnostics, case-specific NBM caching, stricter percentile validation, GRIB-message lookup helpers, obs/data QC summaries, and outlier/percentile-bin diagnostic tables.


This first two cells just import everything we need. **You'll only need to run steps 1 and 2 one time.  After that you can repeat steps 3 and after as many times as you need!**

In [ ]:
#@title 1.  Install required packages with pip
!pip install -q numpy pandas scipy matplotlib seaborn contextily pyproj pygrib netCDF4 cartopy shapely

In [ ]:
#@title 2. Import packages
import numpy as np
from scipy.interpolate import CubicSpline as cs, UnivariateSpline as us
import pandas as pd
from urllib.request import urlretrieve, urlopen
import requests
from datetime import datetime, timedelta
from pathlib import Path as FilePath
import json
from netCDF4 import Dataset
import pygrib
import pyproj
from pyproj import Proj, transform
import os, re, traceback
import sys

import matplotlib
from matplotlib.colors import LinearSegmentedColormap
#from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.axes as maxes
import matplotlib.patheffects as PathEffects
from matplotlib.path import Path
from matplotlib.textpath import TextToPath
import matplotlib.gridspec as gridspec
from matplotlib.font_manager import FontProperties
matplotlib.rcParams['font.sans-serif'] = 'Liberation Sans'
matplotlib.rcParams['font.family'] = "sans-serif"
from matplotlib.cm import get_cmap
import seaborn as sns

from cartopy import crs as ccrs, feature as cfeature
from cartopy.io.shapereader import Reader
import cartopy.io.shapereader as shpreader
from cartopy.feature import ShapelyFeature
import contextily as cx
import itertools

import zipfile

from IPython.display import display, Markdown

import warnings
warnings.filterwarnings("ignore")

The second cell should be run right after the first cell.  Note that these first two cells only need to be run once; then they will work for all other cases until you close or reset this notebook.

In [ ]:

#@title 3. Select Options and Download Obs { display-mode: "form" }
#@markdown If you are using raw obs, you'll need to enter your token for the
#@markdown Synoptic API below.  Below are instructions on how to get a token:
#@markdown https://docs.google.com/document/d/1YuMUYog4J7DpFoEszMmFir4Ehqk9Q0GHG_QhSdrgV9M/edit?usp=sharing
synoptic_token = "" #@param {type:"string"}
#@markdown Which probabilistic element from the NBM?
element = "wind" #@param ["maxt", "mint", "qpf", "qpf48", "qpf72", "wind", "gust", "maxwind", "maxgust", "snow"]
valid_date = "2026-06-14" #@param {type:"date"}
#@markdown Valid hour for QPF ending time or valid-time wind/gust
qpf_valid_time = 18 #@param {type:"slider", min:0, max:18, step:6}
#@markdown APRFC QC'ed gauge data instead of obs (can only check one)?

#@markdown QC gauge data is only for 24hrs valid at 12z of a calendar day.

#@markdown NOHRSC data is only available for CONUS.

#@markdown **Leaving both of these boxes unchecked is the default and results in using raw obs.**
#use_stageiv = True #@param {type:"boolean"}
# Stage IV needs to be fixed due to new NWPS page
# need to use map server at: https://mapservices.weather.noaa.gov/raster/rest/services/obs/rfc_qpe/MapServer

use_stageiv = False
use_nohrsc = False #@param {type:"boolean"}
use_QC_Gauge_Data = False #@param {type:"boolean"}
# NBM v5.0 is now treated as the default operational workflow.
# This constant is used only for plot titles and file names; there is no longer a version selector.
NBM_VERSION_LABEL = "5.0"
#@markdown Pick NBM run time (note: add 1 hour for snow fcsts)
nbm_init_date = "2026-06-13" #@param {type:"date"}
nbm_init_hour = 0 #@param {type:"slider", min:0, max:18, step:6}
#@markdown Where do you want to focus?
region_selection = "CWA" #@param ["WR", "SR", "CR", "ER", "AR", "CONUS", "CWA"]
#@markdown If CWA selected, which one? (i.e. "SLC" for Salt Lake City)
cwa_id = "AFC" #@param {type:"string"}
compare_to = "deterministic" #@param ["obs", "deterministic"]
#@markdown Which obs?
network_selection = "ALL" #@param ["NWS", "RAWS", "NWS+RAWS", "NWS+RAWS+HADS", "ALL", "CUSTOM", "LIST"]
#@markdown If Custom or List selected for network, enter comma separated network IDs (custom) or siteids (list)  WITH NO SPACES here. For help - https://developers.synopticdata.com/about/station-providers/
network_input = "1,2,90,96,122,179,200,3004"#@param {type:"string"}
#@markdown Elevation filter?
elev_filter = False #@param {type:"boolean"}
#@markdown If elevation filter is checked which elevation to you want to filter (ft)?
elev_filter_value = 500 #@param {type:"number"}
#@markdown Would you like to filter out obs above or below this elevation?
filter_above_below = "above" #@param ["above", "below"]
#@markdown Plot CWA boundaries?
cwa_outline = False #@param {type:"boolean"}
county_outline = False #@param {type:"boolean"}
#@markdown Do you want a CSV?
export_csv = True #@param {type:"boolean"}
#@markdown Runtime controls / diagnostics
download_nbm = True #@param {type:"boolean"}
#@markdown If unchecked, notebook will use locally cached NBM subset files only.
strict_percentile_validation = True #@param {type:"boolean"}
export_diagnostics = True #@param {type:"boolean"}
top_n_diagnostics = 10 #@param {type:"slider", min:5, max:25, step:5}
#@markdown Light or dark theme plots?
plot_style = "dark" #@param ["light", "dark"]
#@markdown Custom zoom area for plot (if yes then enter comma seperated lat/lon pairs for your bounding box below (Ex:  custom_southwest = 62.45,-155.55)?
custom_area = True #@param {type:"boolean"}
#@markdown What would you like to name your custom area (mainly for file saving purposes)
custom_area_name = "AnchorageBowl" #@param {type:"string"}
custom_southwest = "60.955924, -151.012975" #@param {type: "string"}
custom_northeast = "62.143834, -147.903844" #@param {type: "string"}
#@markdown Plot cities on map?
plot_cities = True #@param {type:"boolean"}
#@markdown What population threshold would you like to use to thin out cities?
pop_thresh = 5000 #@param

# Base folder for downloaded/subset NBM files and diagnostic outputs.
# Cell 4 replaces NBM_DOWNLOAD_DIR with a case-specific subfolder once the model run/domain are known.
WORK_ROOT = FilePath("nbm")
NBM_DOWNLOAD_DIR = WORK_ROOT
DIAG_ROOT = FilePath("diagnostics")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DIAG_ROOT.mkdir(parents=True, exist_ok=True)
if region_selection == "CONUS":
  region_list = ["WR", "CR", "SR", "ER"]
elif region_selection == "CWA":
  region_list = [cwa_id]
#elif region_selection == "AR":
  #region_list=["AJK","ARH","AFC"]
else:
  region_list = [region_selection]

# Checks for QC Gauge Data
# QC gauge data is only valid for 24-hour QPF. Do not silently change the selected element.
if use_QC_Gauge_Data and element != "qpf":
  print(
      f"QC gauge data is only available for 24-hour QPF. "
      f"You selected element={element!r}. Turning use_QC_Gauge_Data off."
  )
  use_QC_Gauge_Data = False

if use_QC_Gauge_Data and region_selection not in ["AR", "CWA"]:
  print(f"QC gauge data is only available for Alaska offices")
  use_QC_Gauge_Data = False

if use_QC_Gauge_Data and region_selection == "CWA":
  if cwa_id not in ["AFC", "AJK", "AFG"]:
    print(f"QC gauge data is only available for AFC, AJK, and AFG")
    use_QC_Gauge_Data = False

print("Final configuration after compatibility checks:")
print(f"  element = {element}")
print(f"  use_QC_Gauge_Data = {use_QC_Gauge_Data}")
print(f"  use_stageiv = {use_stageiv}")
print(f"  use_nohrsc = {use_nohrsc}")
print(f"  compare_to = {compare_to}")

def cwa_list(input_region):
  region_dict ={"WR":"BYZ,BOI,LKN,EKA,FGZ,GGW,TFX,VEF,LOX,MFR,MTR,MSO,PDT,PSR,PIH,PQR,REV,STO,SLC,SGX,HNX,SEW,OTX,TWC",
              "CR":"ABR,BIS,CYS,LOT,DVN,BOU,DMX,DTX,DDC,DLH,FGF,GLD,GJT,GRR,GRB,GID,IND,JKL,EAX,ARX,ILX,LMK,MQT,MKX,MPX,LBF,APX,IWX,OAX,PAH,PUB,UNR,RIW,FSD,SGF,LSX,TOP,ICT",
              "ER":"ALY,LWX,BGM,BOX,BUF,BTV,CAR,CTP,RLX,CHS,ILN,CLE,CAE,GSP,MHX,OKX,PHI,PBZ,GYX,RAH,RNK,AKQ,ILM",
              "SR":"ABQ,AMA,FFC,EWX,BMX,BRO,CRP,EPZ,FWD,HGX,HUN,JAN,JAX,KEY,MRX,LCH,LZK,LUB,MLB,MEG,MFL,MOB,MAF,OHX,LIX,OUN,SJT,SHV,TAE,TBW,TSA",
              "AR":"AJK,AFG,AFC"}
  if (input_region in ["WR", "CR", "SR", "ER", "AR"]):
    cwas_list = region_dict[input_region]
  else:
    cwas_list = input_region
  return cwas_list

def plot_towns(ax, south, north, west, east, population=5000, resolution='10m', transform=ccrs.PlateCarree(), zorder=3):
    """
    This function will download the 'populated_places' shapefile from
    NaturalEarth, trim the shapefile based on the limits of the provided
    lat & long coords, and then plot the locations and names of the towns
    on a given GeoAxes.

    ax = a pyplot axes object
    south = south lat limit (float)
    north = north lat limit (float)
    west = west long limit (float)
    east = east long limit (float)
    resolution= str. either high res:'10m' or low res: '50m'
    population = minimum population of towns to plot (int)
    transform = a cartopy crs object
    """
    #get town locations
    shp_fn = shpreader.natural_earth(resolution=resolution, category='cultural', name='populated_places')
    shp = Reader(shp_fn)
    xy = [pt.coords[0] for pt in shp.geometries()]
    x, y = list(zip(*xy))

    #get town names
    towns = shp.records()
    names_en = []
    max_population = []
    for town in towns:
        #print(town.attributes)
        names = town.attributes['NAME']
        pop = town.attributes['POP_MAX']
        names_en.append(names)
        max_population.append(pop)
    #print(names_en)
    #create data frame and index by the region of the plot
    all_towns = pd.DataFrame({'names_en': names_en, 'x':x, 'y':y, 'population':max_population})
    #print(all_towns.head())
    region_towns = all_towns[(all_towns.y<north) & (all_towns.y>south)
                           & (all_towns.x>west) & (all_towns.x<east)]
    region_towns = region_towns[region_towns.population > population]
    #print(region_towns.head())
    #plot the locations and labels of the towns in the region
    ax.scatter(region_towns.x.values, region_towns.y.values, c ='white', marker= '.', transform=transform, zorder=zorder)
    transform_mpl = ccrs.PlateCarree()._as_mpl_transform(ax) #this is a work-around to transform xy coords in ax.annotate
    for i, txt in enumerate(region_towns.names_en):
         ax.annotate(txt, (region_towns.x.values[i], region_towns.y.values[i]), xycoords=transform_mpl, color='white')

nbm_init = datetime.strptime(nbm_init_date,'%Y-%m-%d') + timedelta(hours=int(nbm_init_hour))

if element == "maxt":
    nbm_core_valid_hour="00"
    nbm_qmd_valid_hour="06"
    valid_date_start = datetime.strptime(valid_date,'%Y-%m-%d')
    valid_date_end = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(days=1)
    obs_start_hour = "1200"
    obs_end_hour = "0600"
    ob_stat = "maximum"
    valid_end_datetime = valid_date_end + timedelta(hours=(int(obs_end_hour)/100))
    nbm_core_valid_end_datetime = valid_date_end + timedelta(hours=int(nbm_core_valid_hour))
    nbm_qmd_valid_end_datetime = valid_date_end + timedelta(hours=int(nbm_qmd_valid_hour))
    core_init = nbm_init + timedelta(hours = 7)
    nbm_core_fhdelta = nbm_core_valid_end_datetime - core_init

elif element == "mint":
    nbm_core_valid_hour="12"
    nbm_qmd_valid_hour="18"
    valid_date_start = datetime.strptime(valid_date,'%Y-%m-%d')
    valid_date_end = datetime.strptime(valid_date,'%Y-%m-%d')
    obs_start_hour = "0000"
    obs_end_hour = "1800"
    ob_stat = "minimum"
    valid_end_datetime = valid_date_end + timedelta(hours=(int(obs_end_hour)/100))
    nbm_core_valid_end_datetime = valid_date_end + timedelta(hours=int(nbm_core_valid_hour))
    nbm_qmd_valid_end_datetime = valid_date_end + timedelta(hours=int(nbm_qmd_valid_hour))
    core_init = nbm_init + timedelta(hours = 7)
    nbm_core_fhdelta = nbm_core_valid_end_datetime - core_init

elif element == "qpf":
    nbm_core_valid_hour = (str(qpf_valid_time)).zfill(2)
    nbm_valid_hour = (str(qpf_valid_time)).zfill(2)
    nbm_qmd_valid_hour=(str(qpf_valid_time)).zfill(2)
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(hours=int(qpf_valid_time))
    valid_date_start = valid_date - timedelta(hours=24)
    valid_date_end = valid_date
    obs_start_hour = (str(qpf_valid_time)).zfill(2)+"00"
    obs_end_hour = (str(qpf_valid_time)).zfill(2)+"00"
    ob_stat = "total"
    valid_end_datetime = valid_date_end
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_date_end
    nbm_qmd_valid_end_datetime = valid_date_end
    nbm_core_fhdelta = nbm_core_valid_end_datetime - nbm_init

elif element == "qpf48":
    nbm_core_valid_hour = (str(qpf_valid_time)).zfill(2)
    nbm_valid_hour = (str(qpf_valid_time)).zfill(2)
    nbm_qmd_valid_hour=(str(qpf_valid_time)).zfill(2)
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(hours=int(qpf_valid_time))
    valid_date_start = valid_date - timedelta(hours=48)
    valid_date_end = valid_date
    obs_start_hour = (str(qpf_valid_time)).zfill(2)+"00"
    obs_end_hour = (str(qpf_valid_time)).zfill(2)+"00"
    ob_stat = "total"
    valid_end_datetime = valid_date_end
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_date_end
    nbm_qmd_valid_end_datetime = valid_date_end
    nbm_core_fhdelta = nbm_core_valid_end_datetime - nbm_init

elif element == "qpf72":
    nbm_core_valid_hour = (str(qpf_valid_time)).zfill(2)
    nbm_valid_hour = (str(qpf_valid_time)).zfill(2)
    nbm_qmd_valid_hour=(str(qpf_valid_time)).zfill(2)
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(hours=int(qpf_valid_time))
    valid_date_start = valid_date - timedelta(hours=72)
    valid_date_end = valid_date
    obs_start_hour = (str(qpf_valid_time)).zfill(2)+"00"
    obs_end_hour = (str(qpf_valid_time)).zfill(2)+"00"
    ob_stat = "total"
    valid_end_datetime = valid_date_end
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_date_end
    nbm_qmd_valid_end_datetime = valid_date_end
    nbm_core_fhdelta = nbm_core_valid_end_datetime - nbm_init

elif element in ["wind", "gust"]:
    # Valid-time / instantaneous wind or gust percentile.
    # Uses valid_date + qpf_valid_time as the valid datetime.
    nbm_core_valid_hour = str(qpf_valid_time).zfill(2)
    nbm_qmd_valid_hour = str(qpf_valid_time).zfill(2)

    valid_dt = datetime.strptime(valid_date, '%Y-%m-%d') + timedelta(hours=int(qpf_valid_time))

    # Use a small observation window centered on the valid time.
    # For sustained wind, the notebook uses the mean over this window.
    # For gust, it uses the max wind_gust / peak_wind_speed in this window.
    obs_window_minutes = 60
    valid_date_start = valid_dt - timedelta(minutes=obs_window_minutes // 2)
    valid_date_end = valid_dt + timedelta(minutes=obs_window_minutes // 2)

    obs_start_hour = valid_date_start.strftime("%H%M")
    obs_end_hour = valid_date_end.strftime("%H%M")
    ob_stat = "valid_time"

    valid_end_datetime = valid_dt
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_dt
    nbm_qmd_valid_end_datetime = valid_dt
    nbm_core_fhdelta = nbm_core_valid_end_datetime - nbm_init

elif element == "maxwind":
    #nbm_core_valid_hour="06"
    #nbm_valid_hour="06"
    nbm_qmd_valid_hour="06"
    obs_start_hour="0600"
    obs_end_hour="0600"
    ob_stat="maximum"
    valid_date_start = datetime.strptime(valid_date,'%Y-%m-%d')
    valid_date_end = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(days=1)
    valid_end_datetime=valid_date_end + timedelta(hours=(int(obs_end_hour)/100))
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_date_end
    nbm_qmd_valid_end_datetime = valid_date_end + timedelta(hours=int(nbm_qmd_valid_hour))
    nbm_core_fhdelta = valid_end_datetime - nbm_init
    if compare_to == "deterministic":
      raise Exception("FATAL ERROR: You must compare to obs when looking at MAXWIND.  Deterministic data are not available!")
    #valid_date=date.strptime(valid_date,'') + timedelta(hours=)

elif element == "maxgust":
    #nbm_core_valid_hour="06"
    #nbm_valid_hour="06"
    nbm_qmd_valid_hour="06"
    obs_start_hour="0600"
    obs_end_hour="0600"
    ob_stat="maximum"
    valid_date_start = datetime.strptime(valid_date,'%Y-%m-%d')
    valid_date_end = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(days=1)
    valid_end_datetime=valid_date_end + timedelta(hours=(int(obs_end_hour)/100))
    core_init = nbm_init
    nbm_core_valid_end_datetime = valid_date_end
    nbm_qmd_valid_end_datetime = valid_date_end + timedelta(hours=int(nbm_qmd_valid_hour))
    nbm_core_fhdelta = valid_end_datetime - nbm_init
    if compare_to == "deterministic":
      raise Exception("FATAL ERROR: You must compare to obs when looking at MAXWIND.  Deterministic data are not available!")
    #valid_date=date.strptime(valid_date,'') + timedelta(hours=)

elif element == "snow":# or element == "ice":
    nbm_core_valid_hour=(str(qpf_valid_time)).zfill(2)
    nbm_qmd_valid_hour=(str(qpf_valid_time)).zfill(2)
    obs_start_hour=(str(qpf_valid_time)).zfill(2)+"00"
    obs_end_hour = (str(qpf_valid_time)).zfill(2)+"00"
    valid_date = datetime.strptime(valid_date,'%Y-%m-%d') #+ timedelta(hours=int(qpf_valid_time))
    valid_date_start = valid_date - timedelta(hours=24)
    valid_date_end = valid_date
    valid_end_datetime = valid_date_end + timedelta(hours=(int(obs_end_hour)/100))
    ob_stat = "total"
    core_init = nbm_init + timedelta(hours = 1)
    nbm_qmd_valid_end_datetime = valid_date_end + timedelta(hours=int(nbm_qmd_valid_hour))
    nbm_core_valid_end_datetime = nbm_qmd_valid_end_datetime
    nbm_core_fhdelta = nbm_core_valid_end_datetime - core_init
    #if compare_to == "deterministic":
    #  raise Exception("FATAL ERROR: You must compare to obs when looking at snow/ice.  Determinsitic data are not avialable yet.")

current_datetime = datetime.now()

nbm_core_forecasthour = nbm_core_fhdelta.total_seconds() / 3600.
if element == "snow":
  nbm_core_forecasthour_start = nbm_core_forecasthour - 24
elif element == "qpf":
  nbm_core_forecasthour_start = nbm_core_forecasthour - 24
elif element == "qpf48":
  nbm_core_forecasthour_start = nbm_core_forecasthour - 48
elif element == "qpf72":
  nbm_core_forecasthour_start = nbm_core_forecasthour - 72
elif element in ["wind", "gust"]:
  nbm_core_forecasthour_start = nbm_core_forecasthour
else:
  nbm_core_forecasthour_start = nbm_core_forecasthour - 12
nbm_qmd_fhdelta = nbm_qmd_valid_end_datetime - nbm_init
nbm_qmd_forecasthour = nbm_qmd_fhdelta.total_seconds() / 3600.
if element in ["qpf", "maxwind", "maxgust", "snow", "ice24"]:
  nbm_qmd_forecasthour_start = nbm_qmd_forecasthour - 24
elif element == "qpf48":
  nbm_qmd_forecasthour_start = nbm_qmd_forecasthour - 48
elif element == "qpf72":
  nbm_qmd_forecasthour_start = nbm_qmd_forecasthour - 72
elif element in ["wind", "gust"]:
  nbm_qmd_forecasthour_start = nbm_qmd_forecasthour
else:
  nbm_qmd_forecasthour_start = nbm_qmd_forecasthour - 18

# Valid-time wind/gust percentiles are available at 6-hourly forecast steps.
if element in ["wind", "gust"]:
  if int(nbm_qmd_forecasthour) % 6 != 0:
    raise ValueError(
        f"{element} percentiles are only available at 6-hour forecast steps. "
        f"You requested F{int(nbm_qmd_forecasthour):03d}."
    )

# checking to make sure our qpf data selection is valid
if element == 'qpf' and use_QC_Gauge_Data:
  today = datetime.utcnow()
  print(today)
  qpeday = ((today-valid_date_end)+timedelta(days=1)).days
  print('QPE day is day '+str(qpeday))
  if qpeday > 29:
    print("Valid date is outside the range of RFC QPE files...switching to raw obs")
    use_QC_Gauge_Data = False
  if qpf_valid_time != 12:
    print("You have selected a QPF valid time of "+ str(qpf_valid_time))
    print("This is invalid as QC gauge data is only valid at 12z...switching to raw obs!")
    use_QC_Gauge_Data = False


statistics_api = "https://api.synopticdata.com/v2/stations/statistics?"
precipitation_api = "https://api.synopticdata.com/v2/stations/precipitation?"
metadata_api = "https://api.synopticdata.com/v2/stations/metadata?"
timeseries_api = "https://api.synopticdata.com/v2/stations/timeseries?"
# Setup a diting a form selection into a sometctionary for translahing we can pass to mesowest API
network_dict = {"NWS+RAWS+HADS":"&network=1,2,106","NWS+RAWS":"&network=1,2", "NWS":"&network=1", "RAWS": "&network=2", "ALL":"", "CUSTOM": "&network="+network_input, "LIST": "&stid="+network_input}
network_string = network_dict[network_selection]

if element in ["qpf", "qpf48", "qpf72"]:
  cmap = get_cmap('PiYG')
  cmap.set_under(color='red')
  cmap.set_over(color='yellow')
elif element == "snow":
  cmap = get_cmap('cool_r')
  cmap.set_under(color='black')
  cmap.set_over(color='yellow')
else:
  #cmap = 'Spectral'
  cmap = get_cmap('bwr')
  cmap.set_under(color='yellow')
  cmap.set_over(color='black')
if use_stageiv and element=="qpf":
  points_str = f'Stage IV @ {network_selection}'
else:
  points_str = network_selection

if plot_style=="light":
  background_color = '#f7f7f7'
  text_color = '#121212'
  map_land_color = '#FAFAF8'
  map_water_color = '#D4DBDD'
  map_border_color = 'grey'
elif plot_style=="dark":
  background_color = '#272727'
  text_color = 'white'
  map_land_color = '#414143'
  map_water_color = '#272727'
  #map_border_color = '#3B3B3D'
  map_border_color = 'white'


########################################################################################################################
# Reusable functions section                                                                                           #
########################################################################################################################

def ensure_dir(path):
  path = FilePath(path)
  path.mkdir(parents=True, exist_ok=True)
  return path

def cache_path(filename):
  return FilePath(NBM_DOWNLOAD_DIR) / filename

def diagnostic_path(filename):
  """Return a string path for diagnostics; avoids pathlib/matplotlib Path collisions in Colab."""
  return str(FilePath(DIAG_ROOT) / filename)

def check_url_exists(url, timeout=15):
  try:
    response = requests.head(url, timeout=timeout, allow_redirects=True)
    if response.status_code == 405:
      response = requests.get(url, timeout=timeout, stream=True)
    return response.status_code == 200
  except requests.RequestException:
    return False

def show_dataframe(df, title=None, max_rows=20):
  if title:
    display(Markdown(f"### {title}"))
  display(df.head(max_rows))

def project3(lon, lat, prj):
  lon = float(lon)
  lat = float(lat)

  outproj = prj
  inproj = Proj(init='epsg:4326')
  nbm_coords = transform(inproj, outproj, lon, lat)
  coordX = nbm_coords[0]
  coordY = nbm_coords[1]
  #print(f'Lat: {lat}, Y: {coordY} | Lon: {lon}, X: {coordX}')
  return(coordX, coordY)


def ll_to_index(datalons, datalats, loclon, loclat):
  abslat = np.abs(datalats-loclat)
  abslon = np.abs(datalons-loclon)
  c = np.maximum(abslon, abslat)
  latlon_idx_flat = np.argmin(c)
  latlon_idx = np.unravel_index(latlon_idx_flat, datalons.shape)
  return(latlon_idx)


def project_hrap(lon, lat, s4x, s4y):
  lon = float(lon)
  lat = float(lat)

  globe = ccrs.Globe(semimajor_axis=6371200)
  hrap_ccrs = proj = ccrs.Stereographic(central_latitude=90.0,
                          central_longitude=255.0,
                          true_scale_latitude=60.0, globe=globe)
  latlon_ccrs = ccrs.PlateCarree()
  hrap_coords = hrap_ccrs.transform_point(lon,lat,src_crs=latlon_ccrs)
  hrap_idx = ll_to_index(s4x, s4y, hrap_coords[0], hrap_coords[1])

  return hrap_idx

def nohrsc_ll2ij(lon,lat,gridlons,gridlats):
  #for a lat/lon grid
  lon = float(lon)
  lat = float(lat)
  lonidx=(np.abs(lon-gridlons)).argmin()
  latidx=(np.abs(lat-gridlats)).argmin()
  return(latidx,lonidx)

def get_stageiv():
  siv_url = "https://water.weather.gov/precip/downloads/"+valid_date_end.strftime('%Y')+"/"+valid_date_end.strftime('%m')+"/"+valid_date_end.strftime('%d')+"/nws_precip_1day_"+valid_date_end.strftime('%Y%m%d')+"_conus.nc"
  data = urlopen(siv_url).read()
  print(siv_url)
  print(data)
  print(f'Valid date end is: {valid_date_end}')
  nc = Dataset('data', memory=data)
  #with Dataset(siv_file, 'r') as nc:
  stageIV = nc.variables['observation']
  s4x = nc.variables['x']
  s4y = nc.variables['y']
  return stageIV, s4x, s4y

def get_nohrsc():
  nohrsc_url = "https://www.nohrsc.noaa.gov/snowfall_v2/data/"+valid_date_end.strftime('%Y%m')+"/sfav2_CONUS_24h_"+valid_date_end.strftime('%Y%m%d%H')+".nc"
  data = urlopen(nohrsc_url).read()

  nc = Dataset('data',memory=data)
  snow=np.asarray(nc.variables['Data']) #make lon by lat array (original lat by lon)
  snowlat = np.asarray(nc.variables['lat'])
  snowlon = np.asarray(nc.variables['lon'])
  return snow,snowlon,snowlat

def get_qc_data():
    today = datetime.utcnow()
    print(today)
    qpeday = (today-valid_date_end)+timedelta(days=1)
    print(qpeday.days)
    try:
      qcurl = f'https://www.weather.gov/source/aprfc/verification/QPEday{qpeday.days}.csv'
      csv_name = f'{qpeday.days}.csv'
      print(f'QC url is: {qcurl}')
      response = requests.get(qcurl)
      with open(csv_name, 'wb') as f:
          f.write(response.content)
      with open(csv_name, 'r') as fl:
          header = fl.readline().strip()
          fl_date_raw = header.split(' - ')[1]
          fl_date = datetime.strptime(fl_date_raw, '%Y%m%d %Hz')
          print(fl_date)
      if fl_date == valid_date_end:
          qcdata = pd.read_csv(csv_name, skiprows=2, names=['stid', 'obs_qpf'])
      else:
          qcurl = f'https://www.weather.gov/source/aprfc/verification/QPEday{int(qpeday.days-1)}.csv'
          csv_name = f'{int(qpeday.days-1)}.csv'
          print(f'Valid url is: {qcurl}')
          response = requests.get(qcurl)
          with open(csv_name, 'wb') as f:
              f.write(response.content)
          qcdata = pd.read_csv(csv_name, skiprows=2, names=['stid', 'obs_qpf'])
    except Exception as e:
      print("Error retreiving RFC data...exiting the script")
      print(e)
      qcdata = pd.DataFrame()
    return qcdata

def K_to_F(kelvin):
  fahrenheit = 1.8*(kelvin-273)+32.
  return fahrenheit

def mps_to_kts(mps):
  kts = mps * 1.94384
  return kts

def mm_to_in(millimeters):
  inches = millimeters * 0.0393701
  return inches

def meters_to_in(meters):
  inches = meters*39.3701
  return inches

def find_roots(x,y):
  s = np.abs(np.diff(np.sign(y))).astype(bool)
  return x[:-1][s] + np.diff(x)[s]/(np.abs(y[1:][s]/y[:-1][s])+1)


def build_synoptic_timeseries_url(
    *,
    token,
    station_query,
    vars_query,
    start,
    end,
    network_query="",
    units_query="",
    extras_query="",
):
  """
  Build a Synoptic /v2/stations/timeseries URL.

  This is preferred for maxt/mint/maxwind/maxgust because it supports
  exact custom windows such as 12Z-06Z, 00Z-18Z, and 06Z-06Z.
  """
  return (
      timeseries_api
      + f"&token={token}"
      + station_query
      + vars_query
      + f"&start={start}"
      + f"&end={end}"
      + network_query
      + units_query
      + extras_query
  )


def get_timeseries_values(station, variable_base):
  """
  Extract all valid numeric values for a variable from a Synoptic timeseries response.

  Examples of variable_base:
    air_temp
    wind_speed
    wind_gust
    peak_wind_speed
  """
  observations = station.get("OBSERVATIONS", {})

  if not isinstance(observations, dict):
    return np.array([], dtype=float)

  values = []

  for key, raw_values in observations.items():
    if not key.startswith(variable_base):
      continue

    if raw_values is None:
      continue

    if not isinstance(raw_values, list):
      raw_values = [raw_values]

    for value in raw_values:
      if value is None:
        values.append(np.nan)
      else:
        try:
          values.append(float(value))
        except (TypeError, ValueError):
          values.append(np.nan)

  arr = np.array(values, dtype=float)
  return arr[np.isfinite(arr)]


def summarize_timeseries_stat(station, element):
  """
  Calculate the verifying observation value from Synoptic timeseries data
  using the exact valid window requested by the notebook.
  """
  if element == "maxt":
    values = get_timeseries_values(station, "air_temp")
    return np.nanmax(values) if len(values) else np.nan

  if element == "mint":
    values = get_timeseries_values(station, "air_temp")
    return np.nanmin(values) if len(values) else np.nan

  if element == "wind":
    # Valid-time wind: mean sustained wind over the small valid-time window.
    values = get_timeseries_values(station, "wind_speed")
    return mps_to_kts(np.nanmean(values)) if len(values) else np.nan

  if element == "gust":
    # Valid-time gust: maximum reported gust/peak wind in the small valid-time window.
    gust_values = get_timeseries_values(station, "wind_gust")
    peak_values = get_timeseries_values(station, "peak_wind_speed")
    combined = np.concatenate([gust_values, peak_values])
    return mps_to_kts(np.nanmax(combined)) if len(combined) else np.nan

  if element == "maxwind":
    # Period maximum wind.
    values = get_timeseries_values(station, "wind_speed")
    return mps_to_kts(np.nanmax(values)) if len(values) else np.nan

  if element == "maxgust":
    # Period maximum gust.
    gust_values = get_timeseries_values(station, "wind_gust")
    peak_values = get_timeseries_values(station, "peak_wind_speed")
    combined = np.concatenate([gust_values, peak_values])
    return mps_to_kts(np.nanmax(combined)) if len(combined) else np.nan

  return np.nan


def validate_synoptic_response(obs_json, obs_url):
  """
  Raise a clear error when Synoptic returns an API error instead of station data.
  """
  summary = obs_json.get("SUMMARY", {})
  response_code = summary.get("RESPONSE_CODE")

  if response_code != 1 or "STATION" not in obs_json:
    raise RuntimeError(
        "Synoptic API request failed or returned no station data.\n"
        f"Response code: {response_code}\n"
        f"Response message: {summary.get('RESPONSE_MESSAGE')}\n"
        f"URL: {obs_url}"
    )


def download_subset(remote_url, remote_file, local_filename):
  print("   > Downloading a subset of NBM gribs")
  ensure_dir(NBM_DOWNLOAD_DIR)
  local_file = str(cache_path(local_filename))
  if "qmd" in remote_file:
    if element == "maxt":
      if (int(nbm_qmd_forecasthour_start) % 24 == 0) and (int(nbm_qmd_forecasthour) % 24 ==0):
        search_string = f':TMP:2 m above ground:{str(int(int(nbm_qmd_forecasthour_start)/24))}-{str(int(int(nbm_qmd_forecasthour)/24))} day max fcst:'
      else:
        search_string = f':TMP:2 m above ground:{str(int(nbm_qmd_forecasthour_start))}-{str(int(nbm_qmd_forecasthour))} hour max fcst:'
    elif element == "mint":
      if (int(nbm_qmd_forecasthour_start) % 24 == 0) and (int(nbm_qmd_forecasthour) % 24 ==0):
        search_string = f':TMP:2 m above ground:{str(int(int(nbm_qmd_forecasthour_start)/24))}-{str(int(int(nbm_qmd_forecasthour)/24))} day min fcst:'
      else:
        search_string = f':TMP:2 m above ground:{str(int(nbm_qmd_forecasthour_start))}-{str(int(nbm_qmd_forecasthour))} hour min fcst:'
    elif element == "qpf":
      if (int(nbm_qmd_forecasthour_start) % 24 == 0) and (int(nbm_qmd_forecasthour) % 24 ==0):
        search_string = f':APCP:surface:{str(int(int(nbm_qmd_forecasthour_start)/24))}-{str(int(int(nbm_qmd_forecasthour)/24))} day acc fcst:'
      else:
        search_string = f':APCP:surface:{str(int(nbm_qmd_forecasthour_start))}-{str(int(nbm_qmd_forecasthour))} hour acc fcst:'
    elif element == "qpf48":
      if (int(nbm_qmd_forecasthour_start) % 48 == 0) and (int(nbm_qmd_forecasthour) % 48 ==0):
        search_string = f':APCP:surface:{str(int(int(nbm_qmd_forecasthour_start)/48))}-{str(int(int(nbm_qmd_forecasthour)/48))} day acc fcst:'
      else:
        search_string = f':APCP:surface:{str(int(nbm_qmd_forecasthour_start))}-{str(int(nbm_qmd_forecasthour))} hour acc fcst:'
    elif element == "qpf72":
      if (int(nbm_qmd_forecasthour_start) % 72 == 0) and (int(nbm_qmd_forecasthour) % 72 ==0):
        search_string = f':APCP:surface:{str(int(int(nbm_qmd_forecasthour_start)/72))}-{str(int(int(nbm_qmd_forecasthour)/72))} day acc fcst:'
      else:
        search_string = f':APCP:surface:{str(int(nbm_qmd_forecasthour_start))}-{str(int(nbm_qmd_forecasthour))} hour acc fcst:'
    elif element == "wind":
      search_string = f':WIND:10 m above ground:{str(int(nbm_qmd_forecasthour))} hour fcst:'

    elif element == "gust":
      search_string = f':GUST:10 m above ground:{str(int(nbm_qmd_forecasthour))} hour fcst:'

    elif element == "maxwind":
      if (int(nbm_qmd_forecasthour_start) % 24 == 0) and (int(nbm_qmd_forecasthour) % 24 == 0):
        search_string = f':WIND:10 m above ground:{str(int(nbm_qmd_forecasthour_start/24))}-{str(int(nbm_qmd_forecasthour/24))} day max fcst:'
      else:
        search_string = f':WIND:10 m above ground:{str(int(nbm_qmd_forecasthour_start))}-{str(int(nbm_qmd_forecasthour))} hour max fcst:'
    elif element == "maxgust":
        if (int(nbm_qmd_forecasthour_start) % 24 == 0) and (int(nbm_qmd_forecasthour) % 24 == 0):
          search_string = f':GUST:10 m above ground:{str(int(nbm_qmd_forecasthour_start/24))}-{str(int(nbm_qmd_forecasthour/24))} day max fcst:'
        else:
          search_string = f':GUST:10 m above ground:{str(int(nbm_qmd_forecasthour_start))}-{str(int(nbm_qmd_forecasthour))} hour max fcst:'
  elif "core" in remote_file:
    if element == "maxt":
      search_string = f':TMAX:2 m above ground:{str(int(nbm_core_forecasthour_start))}-{str(int(nbm_core_forecasthour))} hour max fcst:'
    elif element == "mint":
      search_string = f':TMIN:2 m above ground:{str(int(nbm_core_forecasthour_start))}-{str(int(nbm_core_forecasthour))} hour min fcst:'
    elif element == "wind":
      search_string = f':WIND:10 m above ground:{str(int(nbm_core_forecasthour))} hour fcst:'
    elif element == "gust":
      search_string = f':GUST:10 m above ground:{str(int(nbm_core_forecasthour))} hour fcst:'
    elif element == "snow":
      search_string = f':ASNOW:surface:{str(int(nbm_core_forecasthour_start))}-{str(int(nbm_core_forecasthour))} hour acc'
  print("Search string = ",search_string)
  print(f"NBM qmd forecast hour start is {nbm_qmd_forecasthour_start}")
  print(f"NBM qmd forecast hour is {nbm_qmd_forecasthour}")
  print(f"NBM core forecast hour start is {nbm_core_forecasthour_start}")
  idx = remote_url+".idx"
  #print("IDX file = " + idx)
  r = requests.get(idx)
  if not r.ok:
    print('     ❌ SORRY! Status Code:', r.status_code, r.reason)
    print(f'      ❌ It does not look like the index file exists: {idx}')

  lines = r.text.split('\n')
  expr = re.compile(search_string)
  expr
  byte_ranges = {}
  for n, line in enumerate(lines, start=1):
    # n is the line number (starting from 1) so that when we call for
    # `lines[n]` it will give us the next line. (Clear as mud??)
    # Use the compiled regular expression to search the line
    #print(">> Searching throgh this line: " + line)
    if expr.search(line):
      # aka, if the line contains the string we are looking for...
      # Get the beginning byte in the line we found
      parts = line.split(':')
      rangestart = int(parts[1])
      # Get the beginning byte in the next line...
      if n+1 < len(lines):
        # ...if there is a next line
        parts = lines[n].split(':')
        rangeend = int(parts[1])
      else:
        # ...if there isn't a next line, then go to the end of the file.
        rangeend = ''

        # Store the byte-range string in our dictionary,
        # and keep the line information too so we can refer back to it.
      byte_ranges[f'{rangestart}-{rangeend}'] = line
      #print(line)
    #else:
      #print(">>>  Could not find search string!")
  #print(">>  Number of items in byteRange:" + str(len(byte_ranges)))
  for i, (byteRange, line) in enumerate(byte_ranges.items()):

    if i == 0:
      # If we are working on the first item, overwrite the existing file.
      curl = f'curl -s --range {byteRange} {remote_url} > {local_file}'
      #print(">>  Adding curl command: " + curl)
    else:
      # If we are working on not the first item, append the existing file.
      curl = f'curl -s --range {byteRange} {remote_url} >> {local_file}'
      #print("Adding curl command: " + curl)
    #print('>>  Parsing line: ' + line)
    try:
      num, byte, date, var, level, forecast, _ = line.split(':')
    except:
      pass
      #print(">>>  Can't get num/byte/etc from this line, so skipping...")

    #print(f'  Downloading GRIB line [{num:>3}]: variable={var}, level={level}, forecast={forecast}')
    #print(f'  Downloading GRIB line: variable={var}, level={level}, forecast={forecast}')
    #print("Running the curl command...")
    os.system(curl)

  if os.path.exists(local_file):
    print(f'      ✅ Success! Searched for [{search_string}] and got [{len(byte_ranges)}] GRIB fields and saved as {local_file}')
    return local_file
  else:
    print(print(f'      ❌ Unsuccessful! Searched for [{search_string}] and did not find anything!'))


########################################################################################################################
# This section for downloading and processing obs                                                                      #
########################################################################################################################
print('Getting obs...')
if custom_area:
  lonmin = float(custom_southwest.split(',')[1])
  latmin = float(custom_southwest.split(',')[0])
  lonmax = float(custom_northeast.split(',')[1])
  latmax = float(custom_northeast.split(',')[0])
obs ={}
for region in region_list:
  if (valid_end_datetime <= current_datetime):
    print("   > Grabbing obs for: ", region)
    #print("List of CWAs: ", cwa_list(region) )
    json_name = "obs/Obs_"+element+"_"+valid_date_start.strftime('%Y%m%d')+obs_start_hour+"_"+valid_date_end.strftime('%Y%m%d')+obs_end_hour+"_"+region+".json"
    if os.path.exists("obs"):
      pass
    else:
      os.mkdir("obs")
    print(f"Element is: {element}")
    # Time-series-based observation windows for temperature and wind.
    # This keeps the observed max/min aligned with the exact NBM valid window.
    if element in ["mint", "maxt"]:
      if custom_area:
        station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
      else:
        station_query = "&cwa=" + cwa_list(region)

      obs_url = build_synoptic_timeseries_url(
          token=synoptic_token,
          station_query=station_query,
          vars_query="&vars=air_temp",
          start=valid_date_start.strftime("%Y%m%d") + obs_start_hour,
          end=valid_date_end.strftime("%Y%m%d") + obs_end_hour,
          network_query=network_string,
          units_query="&units=temp%7Cf",
          extras_query="&obtimezone=utc&status=active",
      )

    elif element == "wind":
      if custom_area:
        station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
      else:
        station_query = "&cwa=" + cwa_list(region)

      obs_url = build_synoptic_timeseries_url(
          token=synoptic_token,
          station_query=station_query,
          vars_query="&vars=wind_speed",
          start=valid_date_start.strftime("%Y%m%d%H%M"),
          end=valid_date_end.strftime("%Y%m%d%H%M"),
          network_query=network_string,
          units_query="&units=speed%7Cmps",
          extras_query="&obtimezone=utc&status=active",
      )

    elif element == "gust":
      if custom_area:
        station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
      else:
        station_query = "&cwa=" + cwa_list(region)

      obs_url = build_synoptic_timeseries_url(
          token=synoptic_token,
          station_query=station_query,
          vars_query="&vars=wind_gust,peak_wind_speed",
          start=valid_date_start.strftime("%Y%m%d%H%M"),
          end=valid_date_end.strftime("%Y%m%d%H%M"),
          network_query=network_string,
          units_query="&units=speed%7Cmps",
          extras_query="&obtimezone=utc&status=active",
      )

    elif element == "maxwind":
      if custom_area:
        station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
      else:
        station_query = "&cwa=" + cwa_list(region)

      obs_url = build_synoptic_timeseries_url(
          token=synoptic_token,
          station_query=station_query,
          vars_query="&vars=wind_speed",
          start=valid_date_start.strftime("%Y%m%d") + obs_start_hour,
          end=valid_date_end.strftime("%Y%m%d") + obs_end_hour,
          network_query=network_string,
          units_query="&units=speed%7Cmps",
          extras_query="&obtimezone=utc&status=active",
      )

    elif element == "maxgust":
      if custom_area:
        station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
      else:
        station_query = "&cwa=" + cwa_list(region)

      obs_url = build_synoptic_timeseries_url(
          token=synoptic_token,
          station_query=station_query,
          vars_query="&vars=wind_gust,peak_wind_speed",
          start=valid_date_start.strftime("%Y%m%d") + obs_start_hour,
          end=valid_date_end.strftime("%Y%m%d") + obs_end_hour,
          network_query=network_string,
          units_query="&units=speed%7Cmps",
          extras_query="&obtimezone=utc&status=active",
      )
    elif element == "qpf":
      if use_stageiv and not use_QC_Gauge_Data:
        api_token = "&token="+synoptic_token
        if custom_area:
          station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
        else:
          station_query = "&cwa="+cwa_list(region)
        api_extras = "&fields=status,latitude,longitude,name,elevation"
        network_query = network_string
        obs_url = metadata_api + api_token + station_query + network_query + api_extras
        stageIV, s4xs, s4ys = get_stageiv()
        s4xs, s4ys = np.meshgrid(s4xs, s4ys)
      elif use_QC_Gauge_Data and not use_stageiv:
        api_token = "&token="+synoptic_token
        if custom_area:
          station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
        else:
          station_query = "&cwa="+cwa_list(region)
        api_extras = "&fields=status,latitude,longitude,name,elevation"
        network_query = network_string
        obs_url = metadata_api + api_token + station_query + network_query + api_extras
        print(f'Obs URL is: {obs_url}')
        qc_df = get_qc_data()
        #print(f'QC data is: {qc_df}')
        is_empty = qc_df.empty
        if is_empty:
          sys.exit()
      elif use_QC_Gauge_Data and use_stageiv:
        print('You cannot use both StageIV AND QC Gauge data silly! Reverting to obs only...')
        api_token = "&token="+synoptic_token
        if custom_area:
          station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
        else:
          station_query = "&cwa="+cwa_list(region)
        api_extras = "&fields=status,latitude,longitude,name,elevation&obtimezone=utc"
        network_query = network_string
        vars_query = "&pmode=totals"
        units_query = "&units=precip|in"
        start_query = "&start="+valid_date_start.strftime('%Y%m%d')+obs_start_hour
        end_query = "&end="+valid_date_end.strftime('%Y%m%d')+obs_end_hour
        obs_url = precipitation_api + api_token + station_query + network_query + vars_query + start_query + end_query + units_query + api_extras
      else:
        api_token = "&token="+synoptic_token
        if custom_area:
          station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
        else:
          station_query = "&cwa="+cwa_list(region)
        api_extras = "&fields=status,latitude,longitude,name,elevation&obtimezone=utc"
        network_query = network_string
        print(network_string)
        vars_query = "&pmode=totals"
        units_query = "&units=precip|in"
        start_query = "&start="+valid_date_start.strftime('%Y%m%d')+obs_start_hour
        end_query = "&end="+valid_date_end.strftime('%Y%m%d')+obs_end_hour
        obs_url = precipitation_api + api_token + station_query + network_query + vars_query + start_query + end_query + units_query + api_extras
    elif element == "qpf48":
      #print("Made it here!")
      api_token = "&token="+synoptic_token
      if custom_area:
        station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
      else:
        station_query = "&cwa="+cwa_list(region)
      api_extras = "&fields=status,latitude,longitude,name,elevation&obtimezone=utc"
      network_query = network_string
      print(network_string)
      vars_query = "&pmode=totals"
      units_query = "&units=precip|in"
      start_query = "&start="+valid_date_start.strftime('%Y%m%d')+obs_start_hour
      end_query = "&end="+valid_date_end.strftime('%Y%m%d')+obs_end_hour
      obs_url = precipitation_api + api_token + station_query + network_query + vars_query + start_query + end_query + units_query + api_extras
    elif element == "qpf72":
      api_token = "&token="+synoptic_token
      if custom_area:
        station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
      else:
        station_query = "&cwa="+cwa_list(region)
      api_extras = "&fields=status,latitude,longitude,name,elevation&obtimezone=utc"
      network_query = network_string
      print(network_string)
      vars_query = "&pmode=totals"
      units_query = "&units=precip|in"
      start_query = "&start="+valid_date_start.strftime('%Y%m%d')+obs_start_hour
      end_query = "&end="+valid_date_end.strftime('%Y%m%d')+obs_end_hour
      obs_url = precipitation_api + api_token + station_query + network_query + vars_query + start_query + end_query + units_query + api_extras
    elif element == "snow":
      if use_nohrsc:
        api_token = "&token="+synoptic_token
        if custom_area:
          station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
        else:
          station_query = "&cwa="+cwa_list(region)
        api_extras = "&fields=status,latitude,longitude,name,elevation"
        network_query = network_string
        obs_url = metadata_api + api_token + station_query + network_query + api_extras
        snow,snowlon,snowlat = get_nohrsc()
        snowlons,snowlats = np.meshgrid(snowlon,snowlat)
      else:
        api_token = "&token="+synoptic_token
        if custom_area:
          station_query = f"&bbox={lonmin},{latmin},{lonmax},{latmax}"
        else:
          station_query = "&cwa="+cwa_list(region)
        api_extras = "&fields=status,latitude,longitude,name,elevation&obtimezone=utc"
        network_query = network_string
        vars_query = "&pmode=totals"
        units_query = "&units=precip|in"
        start_query = "&start="+valid_date_start.strftime('%Y%m%d')+obs_start_hour
        end_query = "&end="+valid_date_end.strftime('%Y%m%d')+obs_end_hour
        obs_url = precipitation_api + api_token + station_query + network_query + vars_query + start_query + end_query + units_query + api_extras
    print("Obs url: " + obs_url)
    if os.path.exists(json_name):
      print ("Deleting old JSON file")
      os.remove(json_name)
      urlretrieve(obs_url, json_name)
    else:
      urlretrieve(obs_url, json_name)

    if os.path.exists(json_name):
        with open(json_name) as json_file:
            obs_json = json.load(json_file)
            print ("Loaded Obs JSON file line 343: " + json_name)
            validate_synoptic_response(obs_json, obs_url)
            obs_lats = []
            obs_lons = []
            obs_value = []
            obs_elev = []
            obs_stid = []
            obs_name = []
            for stn in obs_json["STATION"]:
                # print(stn.encode('utf-8'))
                if stn["STID"] is None:
                  stid = "N0N3"
                else:
                  stid = stn["STID"]
                #print(f'Processing {region} station {stid}')
                name = stn["NAME"]
                if stn["ELEVATION"] and stn["ELEVATION"] is not None:
                  elev = stn["ELEVATION"]
                else:
                  elev = -999
                lat = stn["LATITUDE"]
                lon = stn["LONGITUDE"]
                if float(lon) > -50:
                  continue #bug fix to deal with errant synoptic labs obs in the file
                if element in ["mint", "maxt", "wind", "gust", "maxwind", "maxgust"]:
                  stat = summarize_timeseries_stat(stn, element)

                  if pd.notna(stat):
                    obs_stid.append(str(stid))
                    obs_name.append(str(name))
                    obs_elev.append(float(elev))
                    obs_lats.append(float(lat))
                    obs_lons.append(float(lon))
                    obs_value.append(float(stat))

                elif (element == "qpf"):
                  if (stn["STATUS"] == "ACTIVE"): # and float(stn["LATITUDE"]) < 50.924 and float(stn["LATITUDE"]) > 23.377 and float(stn["LONGITUDE"]) > -125.650 and float(stn["LONGITUDE"]) < -66.008:
                    obs_stid.append(str(stid))
                    obs_name.append(str(name))
                    obs_elev.append(float(elev))
                    obs_lats.append(float(lat))
                    obs_lons.append(float(lon))
                    if use_stageiv and not use_QC_Gauge_Data:
                      coords = project_hrap(lon, lat, s4xs, s4ys)
                      siv_value = float(stageIV[coords])
                      if (siv_value >= 0.01):
                        obs_value.append(siv_value)
                      else:
                        obs_value.append(0.0)
                    elif use_QC_Gauge_Data and not use_stageiv:
                      if stn['STID'] in qc_df['stid'].values.tolist():
                        pcp_amt = qc_df.loc[qc_df['stid'] == stn['STID'], 'obs_qpf'].iloc[0]
                        obs_value.append(float(pcp_amt))
                      else:
                        obs_value.append(np.nan)
                    elif use_QC_Gauge_Data and use_stageiv:
                      if "precipitation" in stn["OBSERVATIONS"]:
                        if "total" in stn["OBSERVATIONS"]["precipitation"][0]:
                           ptotal = stn["OBSERVATIONS"]["precipitation"][0]["total"]
                           if ptotal >= 0.005:
                              obs_value.append(ptotal)
                           else:
                              obs_value.append(0.0)
                        else:
                            obs_value.append(np.nan)
                      else:
                          obs_value.append(np.nan)
                    else:
                      if "precipitation" in stn["OBSERVATIONS"]:
                        if "total" in stn["OBSERVATIONS"]["precipitation"][0]:
                          ptotal = stn["OBSERVATIONS"]["precipitation"][0]["total"]
                          if ptotal >= 0.005:
                            obs_value.append(ptotal)
                          else:
                            obs_value.append(0.0)
                        else:
                          obs_value.append(np.nan)
                      else:
                        obs_value.append(np.nan)
                elif (element == "qpf48"):
                  if (stn["STATUS"] == "ACTIVE"): # and float(stn["LATITUDE"]) < 50.924 and float(stn["LATITUDE"]) > 23.377 and float(stn["LONGITUDE"]) > -125.650 and float(stn["LONGITUDE"]) < -66.008:
                    obs_stid.append(str(stid))
                    obs_name.append(str(name))
                    obs_elev.append(float(elev))
                    obs_lats.append(float(lat))
                    obs_lons.append(float(lon))
                    if "precipitation" in stn["OBSERVATIONS"]:
                        if "total" in stn["OBSERVATIONS"]["precipitation"][0]:
                          ptotal = stn["OBSERVATIONS"]["precipitation"][0]["total"]
                          if ptotal >= 0.005:
                            obs_value.append(ptotal)
                          else:
                            obs_value.append(0.0)
                        else:
                          obs_value.append(np.nan)
                    else:
                      obs_value.append(np.nan)
                elif (element == "qpf72"):
                  if (stn["STATUS"] == "ACTIVE"): # and float(stn["LATITUDE"]) < 50.924 and float(stn["LATITUDE"]) > 23.377 and float(stn["LONGITUDE"]) > -125.650 and float(stn["LONGITUDE"]) < -66.008:
                    obs_stid.append(str(stid))
                    obs_name.append(str(name))
                    obs_elev.append(float(elev))
                    obs_lats.append(float(lat))
                    obs_lons.append(float(lon))
                    if "precipitation" in stn["OBSERVATIONS"]:
                        if "total" in stn["OBSERVATIONS"]["precipitation"][0]:
                          ptotal = stn["OBSERVATIONS"]["precipitation"][0]["total"]
                          if ptotal >= 0.005:
                            obs_value.append(ptotal)
                          else:
                            obs_value.append(0.0)
                        else:
                          obs_value.append(np.nan)
                    else:
                      obs_value.append(np.nan)
                elif (element == "snow"):
                  if stn["STATUS"] == "ACTIVE": # and float(stn["LATITUDE"]) < 50.924)and float(stn["LATITUDE"]) > 23.377 and float(stn["LONGITUDE"]) > -125.650 and float(stn["LONGITUDE"]) < -66.008:
                    obs_stid.append(str(stid))
                    obs_name.append(str(name))
                    obs_elev.append(float(elev))
                    obs_lats.append(float(lat))
                    obs_lons.append(float(lon))
                    if use_nohrsc:
                      coords = nohrsc_ll2ij(lon,lat,snowlon,snowlat)
                      nohrsc_value = meters_to_in(float(snow[coords]))
                      if nohrsc_value >= 0.005:
                        obs_value.append(nohrsc_value)
                      elif nohrsc_value < 0.0:
                        obs_value.append(np.nan)
                      else:
                        obs_value.append(0.0)
                    else:
                      raise Exception("Still not able to process individual snow obs!")
            csv_name = "obs_"+element+"_"+region+".csv"
            obs[region] = pd.DataFrame()
            obs[region]["stid"] = obs_stid
            obs[region]["name"] = obs_name
            obs[region]["elevation"] = obs_elev
            obs[region]["lat"] = obs_lats
            obs[region]["lon"] = obs_lons
            obs[region]["ob_"+element] = obs_value
            obs[region].dropna(inplace=True)
            if elev_filter and filter_above_below == "above":
              obs[region] = obs[region][obs[region]["elevation"] <= elev_filter_value]
            if elev_filter and filter_above_below == "below":
              obs[region] = obs[region][obs[region]["elevation"] >= elev_filter_value]
            obs[region].to_csv(csv_name)
  else:
    print(f'    > Valid Time in the future. Grabbing obs points only for: {region}')
    json_name = "obs/ObsPoints_"+region+"_wcoss.json"
    if os.path.exists(json_name):
      pass
    else:
      if os.path.exists("obs"):
        pass
      else:
        os.mkdir("obs")
      obs_url = "https://api.synopticdata.com/v2/stations/metadata?&token="+synoptic_token+"&cwa="+cwa_list(region)+"&fields=status,latitude,longitude,name,elevation"+network_string
      urlretrieve(obs_url, json_name)
    if os.path.exists(json_name):
      with open(json_name) as json_file:
          obs_json = json.load(json_file)
          print("Loaded Obs JSON file line 427!")
          obs_lats = []
          obs_lons = []
          obs_elev = []
          obs_stid = []
          obs_name = []
          for stn in obs_json["STATION"]:
            # print(stn.encode('utf-8'))
            if stn["STID"] is None:
              stid = "N0N3"
            else:
              stid = stn["STID"]
            #print(f'Processing {region} station {stid}')
            name = stn["NAME"]
            if stn["ELEVATION"] and stn["ELEVATION"] is not None:
              elev = stn["ELEVATION"]
            else:
              elev = -999
            lat = stn["LATITUDE"]
            lon = stn["LONGITUDE"]
            if stn["STATUS"] == "ACTIVE": # and float(stn["LATITUDE"]) != 0. and float(stn["LONGITUDE"]) != 0.:
              obs_stid.append(str(stid))
              obs_name.append(str(name))
              obs_elev.append(float(elev))
              obs_lats.append(float(lat))
              obs_lons.append(float(lon))
          obs[region] = pd.DataFrame()
          obs[region]["stid"] = obs_stid
          obs[region]["name"] = obs_name
          obs[region]["elevation"] = obs_elev
          obs[region]["lat"] = obs_lats
          obs[region]["lon"] = obs_lons
          obs[region]["ob_"+element] = -999
          obs[region].dropna(inplace=True)
          #obs[region].to_csv(csv_name)


# Summarize observation availability after retrieval.
obs_qc_rows = []
for _region, _df in obs.items():
  _row = {"region": _region, "n_rows": len(_df)}
  if not _df.empty:
    _var = "ob_" + element
    _row["n_missing_obs"] = int(_df[_var].isna().sum()) if _var in _df.columns else None
    _row["n_missing_latlon"] = int(_df[["lat", "lon"]].isna().any(axis=1).sum()) if {"lat", "lon"}.issubset(_df.columns) else None
    if "elevation" in _df.columns:
      _row["min_elev_ft"] = round(float(_df["elevation"].min()), 1)
      _row["max_elev_ft"] = round(float(_df["elevation"].max()), 1)
  obs_qc_rows.append(_row)
obs_qc_summary = pd.DataFrame(obs_qc_rows)
if not obs_qc_summary.empty:
  show_dataframe(obs_qc_summary, "Observation availability / QC summary")
  if export_diagnostics:
    obs_qc_summary.to_csv(diagnostic_path(f"obs_qc_summary_{element}_{nbm_init.strftime('%Y%m%d_%H')}.csv"), index=False)


In [ ]:
#@title 4. Download Gribs and Interpolate
########################################################################################################################
# This section downloads and processes the NBM.                                                                        #
########################################################################################################################
if "AR" in region_list or "AJK" in region_list or "ARH" in region_list or "AFC" in region_list or "AFG" in region_list:
  rg="ak"
elif "HFO" in region_list:
  rg="hi"
elif "SJU" in region_list:
  rg="pr"
else:
  rg="co"
print(f'Region list is: {region_list}')
print(f'Region is: {rg}')
print('Getting and processing NBM...')
nbm_init_filen = nbm_init.strftime('%Y%m%d') + "_" + nbm_init.strftime('%H')
nbm_init_filen_core = core_init.strftime('%Y%m%d') + "_" + core_init.strftime('%H')

# Case-specific cache directory keeps different runs/domains/elements from colliding.
NBM_DOWNLOAD_DIR = ensure_dir(WORK_ROOT / nbm_init_filen / rg / element)
DIAG_ROOT = ensure_dir(FilePath("diagnostics") / nbm_init_filen / rg / element)
print(f"NBM cache directory: {NBM_DOWNLOAD_DIR}")
print(f"Diagnostics directory: {DIAG_ROOT}")
# NBM v5.0 is now operational, so use the operational NBM public S3 bucket.
nbm_url_base = "https://noaa-nbm-grib2-pds.s3.amazonaws.com/blend."+nbm_init.strftime('%Y%m%d') \
            +"/"+nbm_init.strftime('%H')+"/"
nbm_url_base_core = "https://noaa-nbm-grib2-pds.s3.amazonaws.com/blend."+core_init.strftime('%Y%m%d') \
            +"/"+core_init.strftime('%H')+"/"
temp_vars = ["maxt","mint"]
if (element in ["qpf", "qpf48", "qpf72"]):
  detr_file = f'blend.t{int(nbm_init_hour):02}z.qmd.f{int(nbm_qmd_forecasthour):03}.{rg}.grib2'
  detr_file_subset = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{rg}.{element}_subset.grib2'
  detr_url = nbm_url_base+"qmd/"+detr_file
  print(f"NBM URL is: {detr_url}")

elif any(te in element for te in temp_vars):
  detr_file = f"blend.t{int(core_init.strftime('%H')):02}z.core.f{int(nbm_core_forecasthour):03}.{rg}.grib2"
  detr_file_subset = f"blend.t{int(core_init.strftime('%H')):02}z.core.{nbm_init_filen_core}f{int(nbm_core_forecasthour):03}.{rg}.{element}_subset.grib2"
  detr_url = nbm_url_base_core+"core/"+detr_file

elif element in ["wind", "gust"]:
  # Valid-time deterministic wind/gust are available in the NBM core file.
  detr_file = f"blend.t{int(core_init.strftime('%H')):02}z.core.f{int(nbm_core_forecasthour):03}.{rg}.grib2"
  detr_file_subset = f"blend.t{int(core_init.strftime('%H')):02}z.core.{nbm_init_filen_core}f{int(nbm_core_forecasthour):03}.{rg}.{element}_subset.grib2"
  detr_url = nbm_url_base_core + "core/" + detr_file

elif element in ["maxwind", "maxgust"]:
  print("MAXWIND/MAXGUST: No deterministic core max-period field available; using NaN deterministic values.")

elif element == "snow":
  #for snow, even for prob/perecntiles, we are dealing with a core file
  detr_file=f"blend.t{int(core_init.strftime('%H')):02}z.core.f{int(nbm_core_forecasthour):03}.{rg}.grib2"
  detr_file_subset = f"blend.t{int(core_init.strftime('%H')):02}z.core.{nbm_init_filen_core}f{int(nbm_core_forecasthour):03}.{rg}.{element}_subset.grib2"
  detr_url = nbm_url_base_core+"core/"+detr_file
  #raise Exception ("Not ready to deal with snow deterministic yet!")
  #print("SNOW: No Core data avaialable!")

if element not in ["maxwind", "maxgust"]:
  detr_subset_path = cache_path(detr_file_subset)
  if detr_subset_path.exists():
    print(f"   > NBM deterministic already exists: {detr_subset_path}")
  elif download_nbm:
    print("   > Getting NBM deterministic")
    print(f"Deterministic url is: {detr_url} and file is {detr_file} and subset is {detr_file_subset}")
    idx_ok = check_url_exists(detr_url + ".idx")
    print(f"   > Deterministic IDX available: {idx_ok}")
    download_subset(detr_url, detr_file, detr_file_subset)
  else:
    raise FileNotFoundError(f"download_nbm is False and cached deterministic subset was not found: {detr_subset_path}")
  nbmd = pygrib.open(str(detr_subset_path))
  if element == "maxt":
    deterministic = nbmd.select(name="Maximum temperature",lengthOfTimeRange=12, stepTypeInternal="max")[0]
    deterministic_array = K_to_F(deterministic.values)
  elif element == "mint":
    deterministic = nbmd.select(name="Minimum temperature",lengthOfTimeRange=12, stepTypeInternal="min")[0]
    deterministic_array = K_to_F(deterministic.values)
  elif element == "qpf":
    deterministic = nbmd.select(name="Total Precipitation",lengthOfTimeRange=24)[-1]
    deterministic_array = mm_to_in(deterministic.values)
  elif element == "qpf48":
    deterministic = nbmd.select(name="Total Precipitation",lengthOfTimeRange=48)[-1]
    deterministic_array = mm_to_in(deterministic.values)

  elif element == "qpf72":
    deterministic = nbmd.select(name="Total Precipitation", lengthOfTimeRange=72)[-1]
    deterministic_array = mm_to_in(deterministic.values)

  elif element == "wind":
    # Valid-time / instantaneous deterministic 10 m wind speed.

    try:
      deterministic = nbmd.select(name="10 metre wind speed")[0]
    except (IndexError, ValueError):
      try:
        deterministic = nbmd.select(name="Instantaneous 10 metre wind speed")[0]
      except (IndexError, ValueError):
        try:
          deterministic = nbmd.select(shortName="10si")[0]
        except (IndexError, ValueError):
          print("Could not find deterministic wind message. Available messages are:")
          nbmd.seek(0)
          for g in nbmd:
            print(
                "name=", getattr(g, "name", None),
                "| shortName=", getattr(g, "shortName", None),
                "| typeOfLevel=", getattr(g, "typeOfLevel", None),
                "| level=", getattr(g, "level", None),
                "| stepRange=", getattr(g, "stepRange", None),
            )
          raise ValueError("No deterministic instantaneous wind message found in downloaded subset.")

    deterministic_array = mps_to_kts(deterministic.values)

  elif element == "gust":
    # Valid-time / instantaneous deterministic 10 m gust.
    # NBM core commonly stores this as:
    #   name      = "Instantaneous 10 metre wind gust"
    #   shortName = "i10fg"

    try:
      deterministic = nbmd.select(name="Instantaneous 10 metre wind gust")[0]
    except (IndexError, ValueError):
      try:
        deterministic = nbmd.select(shortName="i10fg")[0]
      except (IndexError, ValueError):
        print("Could not find deterministic gust message. Available messages are:")
        nbmd.seek(0)
        for g in nbmd:
          print(
              "name=", getattr(g, "name", None),
              "| shortName=", getattr(g, "shortName", None),
              "| typeOfLevel=", getattr(g, "typeOfLevel", None),
              "| level=", getattr(g, "level", None),
              "| stepRange=", getattr(g, "stepRange", None),
          )
        raise ValueError("No deterministic instantaneous gust message found in downloaded subset.")

    deterministic_array = mps_to_kts(deterministic.values)

  elif element == "snow":
    deterministic = nbmd.select(name="unknown", lengthOfTimeRange=24)[7]
    deterministic_array = mm_to_in(deterministic.values)

  nbmlats, nbmlons = deterministic.latlons()
  nbmd.close()

  for region in region_list:
    print("     >> Extracting NBM deterministic")
    point_lats = obs[region]["lat"].values
    point_lons = obs[region]["lon"].values
    detr_values = []
    nbm_fidx = []
    for i in range(0, len(point_lats)):
      coords = ll_to_index(nbmlons, nbmlats, point_lons[i], point_lats[i])
      detr_value = deterministic_array[coords]
      nbm_fidx.append(coords)
      if (element in ["qpf", "qpf48", "qpf72"]) and detr_value < 0.005: #set very light values to zero
        detr_value = 0.0
      detr_values.append(detr_value)
    obs[region]["NBM_fidx"] = nbm_fidx
    obs[region]["NBM_D"] = detr_values
else:
  #set up forecast dataframe for wind/gust, but make determinstic values nan
  print("MAXWIND: Putting station locations into dataframe")
  #fcst[cwa]=pd.DataFrame()
  point_lats = obs[region]["lat"].values
  point_lons = obs[region]["lon"].values
  nbm_fidx = []
  nbmlats=None
  nbmlons=None
  stations = obs[region]["stid"]
  detr_values=np.empty(np.shape(stations))
  detr_values.fill(np.nan)
  obs[region]["NBM_D"] = detr_values

# NBM v5.0 percentile structure.
# Snow keeps the reduced set available in the core file; other probabilistic elements use 0-100.
if element == "snow":
  perc_list = [5,10,25,50,75,90,95]
elif element in ["wind", "gust", "maxwind", "maxgust"]:
  perc_list = list(range(0, 105, 5))
else:
  perc_list = [0,5,10,20,30,40,50,60,70,80,90,95,100]
#perc_list = range(1,100,1)
perc_dict = {
    "maxt": "maxt18p",
    "mint": "mint18p",
    "qpf": "qpf24p",
    "qpf48": "qpf48p",
    "qpf72": "qpf72p",
    "wind": "windp",
    "gust": "gustp",
    "maxwind": "maxwind24p",
    "maxgust": "maxgust24p",
    "snow": "snow24p",
}

if element == "snow":
  perc_file = f"blend.t{int(core_init.strftime('%H')):02}z.core.f{int(nbm_core_forecasthour):03}.{rg}.grib2"
  perc_url = nbm_url_base_core+"core/"+perc_file
else:
  perc_file = f'blend.t{int(nbm_init_hour):02}z.qmd.f{int(nbm_qmd_forecasthour):03}.{rg}.grib2'
  perc_url = nbm_url_base+"qmd/"+perc_file

perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{rg}.{element}_subset.grib2'
print("perc_url=",perc_url)
print("perc_file=",perc_file)
perc_subset_path = cache_path(perc_file_subset)
if perc_subset_path.exists():
  print(f"   > NBM probabilistic already exists: {perc_subset_path}")
elif download_nbm:
  print("   > Getting NBM probabilistic")
  idx_ok = check_url_exists(perc_url + ".idx")
  print(f"   > Probabilistic IDX available: {idx_ok}")
  download_subset(perc_url, perc_file, perc_file_subset)
else:
  raise FileNotFoundError(f"download_nbm is False and cached probabilistic subset was not found: {perc_subset_path}")


def grib_get(g, key, default=None):
    """
    Safely get a pygrib key/attribute.

    pygrib can raise RuntimeError instead of returning the provided
    getattr default when a GRIB key is missing.
    """
    try:
        return getattr(g, key)
    except Exception:
        try:
            return g[key]
        except Exception:
            return default


def build_prob_index(prob_msgs):
    """
    Build lightweight lookup indexes so repeated GRIB selects are faster.

    Uses grib_get() because not every NBM message has lengthOfTimeRange,
    stepTypeInternal, percentileValue, etc.
    """
    index = {}

    for g in prob_msgs:
        name = grib_get(g, "name")
        short_name = grib_get(g, "shortName")
        length = grib_get(g, "lengthOfTimeRange")
        step_type = grib_get(g, "stepTypeInternal")
        percentile = grib_get(g, "percentileValue")
        step_range = grib_get(g, "stepRange")
        level = grib_get(g, "level")

        key_tuples = [
            (name, length, step_type, percentile),
            (name, length, None, percentile),
            (name, None, step_type, percentile),
            (name, None, None, percentile),

            # Short-name fallbacks for instantaneous gust/wind.
            (short_name, length, step_type, percentile),
            (short_name, length, None, percentile),
            (short_name, None, step_type, percentile),
            (short_name, None, None, percentile),

            # Extra fallback including stepRange/level.
            (name, step_range, level, percentile),
            (short_name, step_range, level, percentile),
        ]

        for key_tuple in key_tuples:
            index.setdefault(key_tuple, []).append(g)

    return index


def select_from_prob_msgs(prob_msgs, prob_index=None, **criteria):
    """
    Select GRIB messages using safe metadata access.

    This is more robust than direct pygrib.select() for NBM QMD percentile
    messages because some keys are missing on some messages.
    """
    name = criteria.get("name")
    short_name = criteria.get("shortName")
    length = criteria.get("lengthOfTimeRange")
    step_type = criteria.get("stepTypeInternal")
    percentile = criteria.get("percentileValue")
    step_range = criteria.get("stepRange")
    level = criteria.get("level")

    if prob_index is not None:
        candidate_keys = []

        if name is not None:
            candidate_keys.extend([
                (name, length, step_type, percentile),
                (name, length, None, percentile),
                (name, None, step_type, percentile),
                (name, None, None, percentile),
                (name, step_range, level, percentile),
            ])

        if short_name is not None:
            candidate_keys.extend([
                (short_name, length, step_type, percentile),
                (short_name, length, None, percentile),
                (short_name, None, step_type, percentile),
                (short_name, None, None, percentile),
                (short_name, step_range, level, percentile),
            ])

        for key in candidate_keys:
            if key in prob_index:
                return prob_index[key]

    matches = []

    for g in prob_msgs:
        ok = True

        for key, expected_value in criteria.items():
            actual_value = grib_get(g, key)

            if actual_value != expected_value:
                ok = False
                break

        if ok:
            matches.append(g)

    return matches

all_msgs = pygrib.open(str(perc_subset_path))
# Filter messages that have a percentileValue key
prob_msgs = [
    g for g in all_msgs
    if grib_get(g, "percentileValue") is not None
]
prob_index = build_prob_index(prob_msgs)
available_percentiles = sorted({
    int(grib_get(g, "percentileValue"))
    for g in prob_msgs
    if grib_get(g, "percentileValue") is not None
})
print(f"Filtered {len(prob_msgs)} probabilistic messages out of {all_msgs.messages} total")
print(f"Available percentiles in subset: {available_percentiles}")
missing_percentiles = sorted(set(perc_list) - set(available_percentiles))
extra_percentiles = sorted(set(available_percentiles) - set(perc_list))
percentile_validation_summary = pd.DataFrame([
  {
    "element": element,
    "expected_percentiles": str(list(perc_list)),
    "available_percentiles": str(available_percentiles),
    "missing_percentiles": str(missing_percentiles),
    "extra_percentiles": str(extra_percentiles),
    "strict_validation": strict_percentile_validation,
  }
])
show_dataframe(percentile_validation_summary, "NBM percentile validation")
if export_diagnostics:
  percentile_validation_summary.to_csv(diagnostic_path(f"percentile_validation_{element}_{nbm_init_filen}_{rg}.csv"), index=False)
if missing_percentiles and strict_percentile_validation:
  raise ValueError(f"Missing expected NBM percentiles: {missing_percentiles}. Available: {available_percentiles}")

#for i, g in enumerate(nbmperc):
#    print(f"\nMessage #{i + 1}")
#    print("Name:", g.name)
#    print("Available keys:")
#    for key in g.keys():
#        print(f"  {key}")
print('   > Extracting NBM Probabilistic')
for perc in perc_list:
  print(f'     >> Extracting NBM P{int(perc):01}')
  perc_name = "NBM_P"+str(perc)
  if element == "maxt":
    try:
      percdata = K_to_F(select_from_prob_msgs(prob_msgs, prob_index, name="Time-maximum 2 metre temperature", stepTypeInternal="max", percentileValue=perc)[0].values)
    except IndexError:
      #percdata = K_to_F(select_from_prob_msgs(prob_msgs, prob_index, name="2 metre temperature", stepTypeInternal="max", percentileValue=perc)[0].values)
      percdata = K_to_F(select_from_prob_msgs(prob_msgs, prob_index, name="Maximum temperature at 2 metres since previous post-processing", stepTypeInternal="max", percentileValue=perc)[0].values)
    except IndexError:
      percdata = K_to_F(select_from_prob_msgs(prob_msgs, prob_index, name="2 metre temperature", stepTypeInternal="max", percentileValue=perc)[0].values)
  elif element == "mint":
    try:
      percdata = K_to_F(select_from_prob_msgs(prob_msgs, prob_index, name="Time-minimum 2 metre temperature", stepTypeInternal="min", percentileValue=perc)[0].values)
    except IndexError:
      percdata = K_to_F(select_from_prob_msgs(prob_msgs, prob_index, name="Minimum temperature at 2 metres since previous post-processing", stepTypeInternal="min", percentileValue=perc)[0].values)
    except IndexError:
      percdata = K_to_F(select_from_prob_msgs(prob_msgs, prob_index, name="2 metre temperature", stepTypeInternal="min", percentileValue=perc)[0].values)
  elif element == "qpf":
    percdata = mm_to_in(select_from_prob_msgs(prob_msgs, prob_index, name="Total Precipitation",lengthOfTimeRange=24, percentileValue=perc)[0].values)
  elif element == "qpf48":
    percdata = mm_to_in(select_from_prob_msgs(prob_msgs, prob_index, name="Total Precipitation",lengthOfTimeRange=48, percentileValue=perc)[0].values)
  elif element == "qpf72":
    percdata = mm_to_in(select_from_prob_msgs(prob_msgs, prob_index, name="Total Precipitation",lengthOfTimeRange=72, percentileValue=perc)[0].values)
  elif element == "wind":
    try:
      percinv = select_from_prob_msgs(
          prob_msgs,
          prob_index,
          name="10 metre wind speed",
          percentileValue=perc
      )[0]
    except IndexError:
      percinv = select_from_prob_msgs(
          prob_msgs,
          prob_index,
          shortName="10si",
          percentileValue=perc
      )[0]

    percdata = mps_to_kts(percinv.values)

  elif element == "gust":
    try:
      percinv = select_from_prob_msgs(
          prob_msgs,
          prob_index,
          name="Instantaneous 10 metre wind gust",
          percentileValue=perc
      )[0]
    except IndexError:
      percinv = select_from_prob_msgs(
          prob_msgs,
          prob_index,
          shortName="i10fg",
          percentileValue=perc
      )[0]

    percdata = mps_to_kts(percinv.values)

  elif element == "maxwind":
    # 24-hour maximum 10 m wind percentile.
    try:
      percinv = select_from_prob_msgs(
          prob_msgs,
          prob_index,
          name="10 metre wind speed",
          lengthOfTimeRange=24,
          stepTypeInternal="max",
          percentileValue=perc
      )[0]
    except IndexError:
      percinv = select_from_prob_msgs(
          prob_msgs,
          prob_index,
          name="Time-maximum 10 metre wind speed",
          lengthOfTimeRange=24,
          percentileValue=perc
      )[0]

    percdata = mps_to_kts(percinv.values)

  elif element == "maxgust":
    # 24-hour maximum 10 m gust percentile.
    try:
      percinv = select_from_prob_msgs(
          prob_msgs,
          prob_index,
          name="10 metre wind gust",
          lengthOfTimeRange=24,
          stepTypeInternal="max",
          percentileValue=perc
      )[0]
    except IndexError:
      percinv = select_from_prob_msgs(
          prob_msgs,
          prob_index,
          name="Time-maximum 10 metre wind gust",
          lengthOfTimeRange=24,
          percentileValue=perc
      )[0]

    percdata = mps_to_kts(percinv.values)
  elif element == "snow":
    percinv = select_from_prob_msgs(prob_msgs, prob_index, name="unknown",lengthOfTimeRange=24,percentileValue=perc)[0]
    percdata=meters_to_in(percinv.values)

  if nbmlats is None:
    nbmlats,nbmlons=percinv.latlons()
    for i in range(0,len(point_lats)):
      coords = ll_to_index(nbmlons,nbmlats,point_lons[i],point_lats[i])
      nbm_fidx.append(coords)
    obs[region]["NBM_fidx"] = nbm_fidx
  for region in region_list:
    nbm_coords = obs[region]["NBM_fidx"].values
    perc_values = []
    for i in range(0, len(nbm_coords)):
      perc_value = percdata[nbm_coords[i]]
      perc_values.append(perc_value)
    obs[region][perc_name] = perc_values
all_msgs.close()


########################################################################################################################
# This section creates a distribution curve at each site, and interpolates ob and deterministic to percentile space    #
########################################################################################################################
print('Creating point distribution curves and interpolating...')
start_loc = "NBM_P0"
end_loc = "NBM_P100"
for region in region_list:
  if element == "snow":
    perc_start = obs[region].columns.get_loc("NBM_P5")
    perc_end = obs[region].columns.get_loc("NBM_P95")
    all_percs = obs[region].iloc[:, perc_start:perc_end+1].values
  else:
    perc_start = obs[region].columns.get_loc(start_loc)
    perc_end = obs[region].columns.get_loc(end_loc)
    all_percs = obs[region].iloc[:, perc_start:perc_end+1].values
  var_string = "ob_"+element
  all_obs = obs[region][[var_string]].values
  all_nbmd = obs[region][['NBM_D']].values
  obs_percs = []
  nbmd_percs = []
  for i in range(0,len(all_obs)):
    udf = us(perc_list, all_percs[i,:], bbox=[0,100], ext=0, s=0)
    if np.isnan(all_obs[i]):
      ob_perc = np.nan
    elif all_obs[i] == 0.0 and (element == "qpf" or element == "snow"):
        zo=len(perc_list)-np.count_nonzero(all_percs[i,:])
        if zo == 0:
          ob_perc = -10
        elif zo == 1:
          ob_perc = 10
        else:
          ob_perc = np.random.randint(low=1,high=perc_list[zo-1])
    elif all_obs[i] < udf(0):
      ob_perc = -10
    elif all_obs[i] > udf(100):
      ob_perc = 110
    else:
      ob_perc = find_roots(np.arange(0,101,1), udf(np.arange(0,101,1)) - all_obs[i])
      ob_perc = ob_perc[0].round(1)

    if np.isnan(all_nbmd[i]):
      nbm_perc = np.nan
    elif all_nbmd[i] == 0.0 and (element == "qpf" or element == "snow"):
      zn=len(perc_list)-np.count_nonzero(all_percs[i,:])
      if zn == 0:
        nbm_perc = -10
      elif zn == 1:
        nbm_perc = 10
      else:
        #print(f'zn is: {zn} and zo -1 is : {zo-1}')
        #print(f'percent list is: {perc_list}')
        nbm_perc = np.random.randint(low=1,high=perc_list[zn-1])
        #print(f' NBM percentile is: {nbm_perc}')
    elif all_nbmd[i] < udf(0):
      nbm_perc = -10
    elif all_nbmd[i] > udf(100):
      nbm_perc = 110
    else:
      nbm_perc = find_roots(np.arange(0,101,1), udf(np.arange(0,101,1)) - all_nbmd[i])
      nbm_perc = nbm_perc[0].round(1)

    if np.isnan(ob_perc):
      obs_percs.append(ob_perc)
    else:
      obs_percs.append(int(ob_perc))
    if np.isnan(nbm_perc):
      nbmd_percs.append(nbm_perc)
    else:
      nbmd_percs.append(int(nbm_perc))
  obs[region]["ob_perc"] = obs_percs
  obs[region]["NBMd_perc"] = nbmd_percs
  if export_csv:
    csv_name = f"obs_and_percs_{element}_{nbm_init.strftime('%Y%m%d')}_{valid_end_datetime.strftime('%Y%m%d')}_{region}.csv"
    obs[region].to_csv(csv_name)
    if export_diagnostics:
      obs[region].to_csv(diagnostic_path(csv_name))
    print(f'   > Created and saved {csv_name}')


In [ ]:
#@title 5. Run Summary and Diagnostics
# This cell summarizes exactly what was processed before plotting.
if compare_to == "obs":
  diagnostic_compare_var = "ob_perc"
  diagnostic_compare_label = "Observed value in NBM percentile space"
elif compare_to == "deterministic":
  diagnostic_compare_var = "NBMd_perc"
  diagnostic_compare_label = "Deterministic/mean forecast in NBM percentile space"
else:
  diagnostic_compare_var = "ob_perc"
  diagnostic_compare_label = "Percentile space"

run_summary = pd.DataFrame([{
  "NBM version": f"v{NBM_VERSION_LABEL} operational",
  "element": element,
  "compare_to": compare_to,
  "regions": ",".join(region_list),
  "domain": rg,
  "NBM init": nbm_init.strftime("%Y-%m-%d %HZ"),
  "valid start": valid_date_start.strftime("%Y-%m-%d %HZ"),
  "valid end": valid_end_datetime.strftime("%Y-%m-%d %HZ"),
  "QMD forecast hour": int(nbm_qmd_forecasthour),
  "cache dir": str(NBM_DOWNLOAD_DIR),
  "obs source": "APRFC QC gauge" if use_QC_Gauge_Data else ("NOHRSC" if use_nohrsc else ("Stage IV" if use_stageiv else network_selection)),
  "percentiles": str(list(perc_list)),
}])
show_dataframe(run_summary, "Run summary")

all_diagnostic_frames = []
for _region in region_list:
  if _region not in obs or obs[_region].empty:
    continue
  _df = obs[_region].copy()
  _df["region"] = _region
  all_diagnostic_frames.append(_df)

if all_diagnostic_frames:
  diagnostic_df = pd.concat(all_diagnostic_frames, ignore_index=True)
  percentile_bins = [-np.inf, 0, 10, 25, 75, 90, 100, np.inf]
  percentile_labels = ["below P0", "P0-P10", "P10-P25", "P25-P75", "P75-P90", "P90-P100", "above P100"]
  diagnostic_df["percentile_bin"] = pd.cut(
      diagnostic_df[diagnostic_compare_var],
      bins=percentile_bins,
      labels=percentile_labels,
      include_lowest=True,
  )
  bin_summary = (
      diagnostic_df.groupby("percentile_bin", observed=False)
      .size()
      .reset_index(name="count")
  )
  bin_summary["percent_of_points"] = (100 * bin_summary["count"] / max(len(diagnostic_df), 1)).round(1)
  show_dataframe(bin_summary, "Percentile-bin summary")

  metrics = pd.DataFrame([{
      "n_points": len(diagnostic_df),
      "mean_percentile": round(float(diagnostic_df[diagnostic_compare_var].mean()), 1),
      "median_percentile": round(float(diagnostic_df[diagnostic_compare_var].median()), 1),
      "pct_below_p10": round(float((diagnostic_df[diagnostic_compare_var] <= 10).mean() * 100), 1),
      "pct_between_p25_p75": round(float(diagnostic_df[diagnostic_compare_var].between(25, 75).mean() * 100), 1),
      "pct_above_p90": round(float((diagnostic_df[diagnostic_compare_var] >= 90).mean() * 100), 1),
      "pct_outside_p5_p95": round(float(((diagnostic_df[diagnostic_compare_var] <= 5) | (diagnostic_df[diagnostic_compare_var] >= 95)).mean() * 100), 1),
  }])
  show_dataframe(metrics, "Quick verification metrics")

  value_cols = [c for c in ["region", "stid", "name", "lat", "lon", "elevation", "ob_" + element, "NBM_D", diagnostic_compare_var, "percentile_bin"] if c in diagnostic_df.columns]
  lowest_sites = diagnostic_df.sort_values(diagnostic_compare_var, ascending=True).head(top_n_diagnostics)[value_cols]
  highest_sites = diagnostic_df.sort_values(diagnostic_compare_var, ascending=False).head(top_n_diagnostics)[value_cols]
  show_dataframe(lowest_sites, f"Lowest {top_n_diagnostics} percentile placements")
  show_dataframe(highest_sites, f"Highest {top_n_diagnostics} percentile placements")

  if export_diagnostics:
    run_summary.to_csv(diagnostic_path(f"run_summary_{element}_{nbm_init_filen}_{rg}.csv"), index=False)
    bin_summary.to_csv(diagnostic_path(f"percentile_bin_summary_{element}_{nbm_init_filen}_{rg}.csv"), index=False)
    metrics.to_csv(diagnostic_path(f"quick_metrics_{element}_{nbm_init_filen}_{rg}.csv"), index=False)
    lowest_sites.to_csv(diagnostic_path(f"lowest_percentile_sites_{element}_{nbm_init_filen}_{rg}.csv"), index=False)
    highest_sites.to_csv(diagnostic_path(f"highest_percentile_sites_{element}_{nbm_init_filen}_{rg}.csv"), index=False)
    case_config = {
      "nbm_version": NBM_VERSION_LABEL,
      "element": element,
      "compare_to": compare_to,
      "regions": region_list,
      "domain": rg,
      "nbm_init": nbm_init.strftime("%Y-%m-%d %HZ"),
      "valid_start": valid_date_start.strftime("%Y-%m-%d %HZ"),
      "valid_end": valid_end_datetime.strftime("%Y-%m-%d %HZ"),
      "qmd_forecast_hour": int(nbm_qmd_forecasthour),
      "cache_dir": str(NBM_DOWNLOAD_DIR),
      "diagnostics_dir": str(DIAG_ROOT),
      "percentiles": list(perc_list),
    }
    with open(diagnostic_path(f"case_config_{element}_{nbm_init_filen}_{rg}.json"), "w") as _f:
      json.dump(case_config, _f, indent=2)
else:
  print("No diagnostic dataframe could be created because no observations were available.")


In [ ]:
#@title 6. Generate Plot
########################################################################################################################
# Finally, this section makes our plot.
########################################################################################################################

print("Making plot (almost done!)...")

# --------------------------------------------------------------------------------------------------
# Element/product labels
# --------------------------------------------------------------------------------------------------
# obs_label  = observed element label used in plot title
# prob_label = NBM probabilistic product label used in plot title
# det_label  = deterministic comparison label used when compare_to == "deterministic"
# --------------------------------------------------------------------------------------------------

ELEMENT_LABELS = {
    "maxt": {
        "obs_label": "Max T",
        "prob_label": "PMaxT",
        "det_label": "Deterministic Max T",
    },
    "mint": {
        "obs_label": "Min T",
        "prob_label": "PMinT",
        "det_label": "Deterministic Min T",
    },
    "qpf": {
        "obs_label": "QPF",
        "prob_label": "PQPF",
        "det_label": "pMean",
    },
    "qpf48": {
        "obs_label": "48hr QPF",
        "prob_label": "48hr PQPF",
        "det_label": "pMean",
    },
    "qpf72": {
        "obs_label": "72hr QPF",
        "prob_label": "72hr PQPF",
        "det_label": "pMean",
    },
    "wind": {
        "obs_label": "Wind",
        "prob_label": "PWind",
        "det_label": "Deterministic Wind",
    },
    "gust": {
        "obs_label": "Gust",
        "prob_label": "PGust",
        "det_label": "Deterministic Gust",
    },
    "maxwind": {
        "obs_label": "Max Wind",
        "prob_label": "Prob Max Wind",
        "det_label": "Deterministic Max Wind",
    },
    "maxgust": {
        "obs_label": "Max Gust",
        "prob_label": "Prob Max Gust",
        "det_label": "Deterministic Max Gust",
    },
    "snow": {
        "obs_label": "Snow Acc",
        "prob_label": "Prob Snow Acc",
        "det_label": "Deterministic Snow Acc",
    },
}

if element not in ELEMENT_LABELS:
    raise ValueError(
        f"Unsupported element for plotting: {element}. "
        f"Supported elements are: {list(ELEMENT_LABELS.keys())}"
    )

element_label = ELEMENT_LABELS[element]["obs_label"]
prob_label = ELEMENT_LABELS[element]["prob_label"]

if compare_to == "obs":
    compare_var = "ob_perc"
    compare_element = "Obs"
elif compare_to == "deterministic":
    compare_var = "NBMd_perc"
    compare_element = ELEMENT_LABELS[element]["det_label"]
else:
    raise ValueError(f"Unsupported compare_to option: {compare_to}")

print(
    f"Plotting element={element}, "
    f"element_label={element_label}, "
    f"prob_label={prob_label}, "
    f"compare_to={compare_to}, "
    f"compare_element={compare_element}"
)

matplotlib.rc("axes", facecolor=background_color, edgecolor=text_color)

# --------------------------------------------------------------------------------------------------
# Valid/init time labels
# --------------------------------------------------------------------------------------------------

# Use the actual NBM QMD valid end datetime for valid-time products.
# This is especially important for wind/gust, which are available at 6-hourly
# valid times rather than as daily max/min windows.
if element in ["qpf", "qpf48", "qpf72", "snow", "wind", "gust"]:
    valid_datetime = nbm_qmd_valid_end_datetime
    fig_valid_date = nbm_qmd_valid_end_datetime.strftime("%Y%m%d_%HZ")
    valid_title = nbm_qmd_valid_end_datetime.strftime("%HZ %a %m-%d-%Y")
else:
    valid_datetime = datetime.strptime(valid_date, "%Y-%m-%d")
    fig_valid_date = valid_datetime.strftime("%Y%m%d")
    valid_title = valid_datetime.strftime("%a %m-%d-%Y")

if element == "snow":
    nbm_init_title = core_init.strftime("%HZ %m-%d-%Y")
else:
    nbm_init_title = nbm_init.strftime("%HZ %m-%d-%Y")

# Forecast-hour label for valid-time wind/gust and deterministic comparisons.
try:
    forecast_hour_label = f"F{int(nbm_qmd_forecasthour):03d}"
except Exception:
    forecast_hour_label = ""

if element in ["wind", "gust"] and forecast_hour_label:
    valid_title = f"{valid_title} ({forecast_hour_label})"


def flip(items, ncol):
    return itertools.chain(*[items[i::ncol] for i in range(ncol)])


def safe_legend_bins(point_data, min_bins=1, max_bins=10):
    """
    Build a safe integer number of legend bins from the plotted percentile range.
    Prevents zero-bin legends when all plotted values are similar.
    """
    point_data = np.asarray(point_data, dtype=float)
    point_data = point_data[np.isfinite(point_data)]

    if len(point_data) == 0:
        return min_bins

    spread = abs(np.nanmax(point_data) - np.nanmin(point_data))
    numcols = int(spread // 10) + 1
    return max(min_bins, min(numcols, max_bins))

# Predefined CWA plotting extents.
# Format: [west, east, south, north]
CWA_EXTENTS = {
    # Southeast Alaska / Juneau CWA
    "AJK": [-141.8, -129.0, 53.5, 60.5],

    # Anchorage / Southcentral and Southwest Alaska
    "AFC": [-170.0, -135.0, 54.0, 64.5],

    # Fairbanks / Northern Alaska
    "AFG": [-180.0, -130.0, 62.0, 72.5],
}


def get_obs_extent(df, pad_lon=0.75, pad_lat=0.5):
    """
    Fallback extent based on observation points.
    """
    clean_df = df.dropna(subset=["lat", "lon"]).copy()

    if clean_df.empty:
        raise ValueError("Cannot calculate map extent because obs dataframe has no valid lat/lon points.")

    west = clean_df["lon"].min() - pad_lon
    east = clean_df["lon"].max() + pad_lon
    south = clean_df["lat"].min() - pad_lat
    north = clean_df["lat"].max() + pad_lat

    return [west, east, south, north]


def get_cwa_extent(cwa_id, obs_df=None):
    """
    Return a stable plotting extent for known CWAs.
    Falls back to obs-based extent if the CWA is not predefined.
    """
    cwa_key = cwa_id.upper()

    if cwa_key in CWA_EXTENTS:
        return CWA_EXTENTS[cwa_key]

    if obs_df is not None:
        print(f"No predefined extent for {cwa_key}; using obs-based extent.")
        return get_obs_extent(obs_df)

    raise ValueError(f"No predefined extent for {cwa_key} and no obs dataframe supplied.")
# --------------------------------------------------------------------------------------------------
# CONUS multipanel plot
# --------------------------------------------------------------------------------------------------

if region_selection == "CONUS":
    dataframeid = "CONUS"

    west = -125.650
    south = 23.377
    east = -66.008
    north = 50.924
    width_ratios = [7, 3, 3, 3]
    lloc = "lower right"

    fig = plt.figure(
        constrained_layout=True,
        figsize=(16, 9),
        facecolor=background_color,
        frameon=True,
        dpi=150,
    )

    grid = fig.add_gridspec(
        4,
        4,
        width_ratios=width_ratios,
        hspace=0.2,
        wspace=0.2,
        left=0.1,
        right=0.9,
    )

    fig.text(
        0.30,
        0.885,
        f"{region_selection} {element_label} {compare_element} "
        f"in NBM v{NBM_VERSION_LABEL} {prob_label} Percentile Space",
        horizontalalignment="center",
        weight="bold",
        fontsize=25,
        color=text_color,
    )

    fig.text(
        0.30,
        0.855,
        f"Valid: {valid_title}  |  NBM Init: {nbm_init_title}  |  Points: {points_str}",
        horizontalalignment="center",
        fontsize=16,
        color=text_color,
    )

    ax1 = fig.add_subplot(grid[:, :-2], projection=ccrs.Mercator(globe=None))
    ax2 = fig.add_subplot(grid[0, 2])
    ax3 = fig.add_subplot(grid[0, 3])
    ax4 = fig.add_subplot(grid[1, 2])
    ax5 = fig.add_subplot(grid[1, 3])
    ax6 = fig.add_subplot(grid[2:, 2:])

    conus_df = pd.concat([obs["WR"], obs["CR"], obs["ER"], obs["SR"]])

    lats = conus_df["lat"].values
    lons = conus_df["lon"].values
    point_data = conus_df[compare_var].values

    mean = conus_df[compare_var].mean()
    median = conus_df[compare_var].median()

    proj = ccrs.PlateCarree()

    ax1.set_anchor("S")
    ax1.set_extent([west, east, south, north], crs=proj)
    ax1.add_feature(cfeature.OCEAN, edgecolor="none", facecolor=map_water_color, zorder=-2)
    ax1.add_feature(
        cfeature.NaturalEarthFeature(
            "physical",
            "land",
            "50m",
            edgecolor="none",
            facecolor=map_land_color,
            zorder=-1,
        )
    )
    ax1.add_feature(
        cfeature.NaturalEarthFeature(
            "physical",
            "lakes",
            "10m",
            edgecolor="none",
            facecolor=map_water_color,
            zorder=0,
        )
    )
    ax1.add_feature(cfeature.BORDERS, edgecolor=map_border_color, facecolor="none", linewidth=2, zorder=1)
    ax1.add_feature(
        cfeature.NaturalEarthFeature(
            "cultural",
            "admin_1_states_provinces_lines",
            "50m",
            edgecolor=map_border_color,
            facecolor="none",
            linewidth=1,
            zorder=2,
        )
    )

    scatter = ax1.scatter(
        lons,
        lats,
        c=point_data,
        cmap=cmap,
        s=45,
        transform=proj,
        vmin=0.0,
        vmax=100.0,
    )

    if plot_cities:
        plot_towns(ax1, south, north, west, east, population=pop_thresh)

    numcols = safe_legend_bins(point_data)
    legend1 = ax1.legend(
        *scatter.legend_elements(num=numcols),
        loc=lloc,
        title=f"{compare_element}\nRank",
        fancybox=True,
    )

    plt.setp(legend1.get_title(), multialignment="center", color=text_color)

    for text in legend1.get_texts():
        text.set_color(text_color)

    ax1.add_artist(legend1)

    ax1.add_feature(
        cfeature.NaturalEarthFeature(
            "cultural",
            "admin_1_states_provinces_lines",
            "110m",
            edgecolor="gray",
            facecolor="none",
        )
    )

    if cwa_outline:
        try:
            if not os.path.exists("shp/w_22mr22.shp"):
                cwa_url = "https://www.weather.gov/source/gis/Shapefiles/WSOM/w_22mr22.zip"
                if not os.path.exists("shp"):
                    os.mkdir("shp")
                urlretrieve(cwa_url, "shp/nws_cwa_outlines.zip")

                with zipfile.ZipFile("shp/nws_cwa_outlines.zip", "r") as zip_ref:
                    zip_ref.extractall("shp")

            cwa_feature = ShapelyFeature(
                Reader("shp/w_22mr22.shp").geometries(),
                ccrs.PlateCarree(),
                edgecolor="grey",
                facecolor="none",
                linewidth=0.5,
                linestyle=":",
                zorder=3,
            )
            ax1.add_feature(cwa_feature)

        except Exception:
            print("   > Aw shucks, no CWA boundaries for you. Sorry bout that.")

    if county_outline:
        try:
            if not os.path.exists("shp/c_08mr23.shp"):
                county_url = "https://www.weather.gov/source/gis/Shapefiles/County/c_08mr23.zip"
                if not os.path.exists("shp"):
                    os.mkdir("shp")
                urlretrieve(county_url, "shp/counties.zip")

                with zipfile.ZipFile("shp/counties.zip", "r") as zip_ref:
                    zip_ref.extractall("shp")

            cty_feature = ShapelyFeature(
                Reader("shp/c_08mr23.shp").geometries(),
                ccrs.PlateCarree(),
                edgecolor="white",
                facecolor="none",
                linewidth=1.0,
                linestyle="--",
                zorder=3,
            )
            ax1.add_feature(cty_feature)

        except Exception:
            print("   > Cannot plot county boundaries.")

    # Region histograms
    for ax, region_key, region_name in [
        (ax2, "WR", "Western Region"),
        (ax3, "CR", "Central Region"),
        (ax4, "ER", "Eastern Region"),
        (ax5, "SR", "Southern Region"),
    ]:
        region_mean = obs[region_key][compare_var].mean()
        region_median = obs[region_key][compare_var].median()

        ax.set_anchor("N")
        sns.histplot(
            data=obs[region_key],
            x=compare_var,
            ax=ax,
            kde=True,
            bins=range(0, 110, 10),
            color="steelblue",
            edgecolor="lightgrey",
        )

        ax.set_xlabel(region_name, color=text_color, fontsize=12)
        ax.axvline(region_mean, color="salmon", linestyle="--", label="Mean")
        ax.axvline(region_median, color="mediumaquamarine", linestyle="-", label="Median")
        ax.grid(False)

        for tick in ax.get_xticklabels():
            tick.set_color(text_color)

        for tick in ax.get_yticklabels():
            tick.set_color(text_color)

        ax.tick_params(axis="y", labelsize=8, color=text_color)

        legend = ax.legend()

        for text in legend.get_texts():
            text.set_color(text_color)

        ax.set(ylabel=None)

    # CONUS aggregate histogram
    ax6.set_anchor("NC")
    sns.histplot(
        data=point_data,
        ax=ax6,
        kde=True,
        bins=range(0, 110, 10),
        color="steelblue",
        edgecolor="lightgrey",
    )

    ax6.set_xlabel(
        f"{compare_element} in NBM {prob_label} Percentile Bins",
        color=text_color,
        fontsize=12,
    )

    ax6.axvline(mean, color="salmon", linestyle="--", label="Mean")
    ax6.axvline(median, color="mediumaquamarine", linestyle="-", label="Median")
    ax6.grid(False)

    for tick in ax6.get_xticklabels():
        tick.set_color(text_color)

    for tick in ax6.get_yticklabels():
        tick.set_color(text_color)

    ax6.tick_params(axis="y", labelsize=8, color=text_color)

    legend6 = ax6.legend()

    for text in legend6.get_texts():
        text.set_color(text_color)

    ax6.set(ylabel=None)


# --------------------------------------------------------------------------------------------------
# Regional / CWA / custom-area two-panel plot
# --------------------------------------------------------------------------------------------------

else:
    if not custom_area:
        if region_selection == "AR":
            west = -179.00
            south = 52.00
            east = -129.00
            north = 72.00
            width, height = (16, 7)
            width_ratios = [9, 7]
            lloc = "lower left"

        if region_selection == "WR":
            west = -126.917
            south = 30.586
            east = -102.740
            north = 49.755
            width, height = (16, 9)
            width_ratios = [9, 8]
            lloc = "lower right"

        if region_selection == "CR":
            west = -111.534
            south = 33.295
            east = -81.723
            north = 49.755
            width, height = (16, 7)
            width_ratios = [9, 7]
            lloc = "lower center"

        if region_selection == "ER":
            west = -86.129
            south = 31.223
            east = -66.465
            north = 47.676
            width, height = (16, 7.25)
            width_ratios = [6.9, 9.5]
            lloc = "lower right"

        if region_selection == "SR":
            west = -109.758
            south = 23.313
            east = -79.247
            north = 36.899
            width, height = (16, 5.6)
            width_ratios = [10, 6]
            lloc = "lower center"

        if region_selection == "CWA":
          dataframeid = cwa_id

          west, east, south, north = get_cwa_extent(
              cwa_id,
              obs_df=obs.get(dataframeid)
          )

          width, height = (16, 9)
          ratioxy = 16.0 / 9.0
          width_ratios = [ratioxy, 1]
          lloc = "center right"

    else:
        west = float(custom_southwest.split(",")[1])
        south = float(custom_southwest.split(",")[0])
        east = float(custom_northeast.split(",")[1])
        north = float(custom_northeast.split(",")[0])
        width, height = (16, 7)
        width_ratios = [9, 7]
        lloc = "lower left"

    fig = plt.figure(
        constrained_layout=True,
        figsize=(width, height),
        facecolor=background_color,
        frameon=True,
        dpi=150,
    )

    if region_selection == "CWA":
        dataframeid = cwa_id
    else:
        dataframeid = region_selection

    grid = fig.add_gridspec(
        1,
        2,
        hspace=0.2,
        width_ratios=width_ratios,
        height_ratios=[1],
        wspace=0.2,
    )

    ax1 = fig.add_subplot(grid[0, 0], projection=ccrs.Mercator())
    ax2 = fig.add_subplot(grid[0, 1])

    if not custom_area:
        plot_area_label = dataframeid
    else:
        plot_area_label = custom_area_name

    fig.text(
        0.5,
        1.05,
        f"{plot_area_label} {element_label} {compare_element} "
        f"in NBM v{NBM_VERSION_LABEL} {prob_label} Percentile Space",
        horizontalalignment="center",
        verticalalignment="bottom",
        weight="bold",
        fontsize=20,
        color=text_color,
    )

    fig.text(
        0.5,
        1.05,
        f"Valid: {valid_title} | NBM Init: {nbm_init_title} | Points: {points_str}",
        horizontalalignment="center",
        verticalalignment="top",
        fontsize=16,
        color=text_color,
    )

    lats = obs[dataframeid]["lat"].values
    lons = obs[dataframeid]["lon"].values
    point_data = obs[dataframeid][compare_var].values

    mean = obs[dataframeid][compare_var].mean()
    median = obs[dataframeid][compare_var].median()

    proj = ccrs.PlateCarree()
    numcols = safe_legend_bins(point_data)

    ax1.set_anchor("N")
    ax1.set_facecolor(background_color)
    ax1.set_extent([west, east, south, north], crs=proj)

    ax1.add_feature(cfeature.OCEAN, edgecolor="none", facecolor=map_water_color, zorder=-2)
    ax1.add_feature(
        cfeature.NaturalEarthFeature(
            "physical",
            "land",
            "50m",
            edgecolor="none",
            facecolor=map_land_color,
            zorder=-1,
        )
    )
    ax1.add_feature(
        cfeature.NaturalEarthFeature(
            "physical",
            "lakes",
            "10m",
            edgecolor="none",
            facecolor=map_water_color,
            zorder=0,
        )
    )
    ax1.add_feature(cfeature.BORDERS, edgecolor=map_border_color, facecolor="none", linewidth=2, zorder=2)
    ax1.add_feature(
        cfeature.NaturalEarthFeature(
            "cultural",
            "admin_1_states_provinces_lines",
            "50m",
            edgecolor=map_border_color,
            facecolor="none",
            linewidth=1,
            zorder=5,
        )
    )

    scatter = ax1.scatter(
        lons,
        lats,
        c=point_data,
        cmap=cmap,
        s=45,
        transform=proj,
        zorder=2,
        vmin=0.0,
        vmax=100.0,
    )

    if plot_cities:
        plot_towns(ax1, south, north, west, east, population=pop_thresh)

    if region_selection in ("CR", "SR"):
        handles, labels = scatter.legend_elements(num=numcols)
        legend1 = ax1.legend(
            flip(handles, 6),
            flip(labels, 6),
            ncol=6,
            loc=lloc,
            title=f"{compare_element} in NBM {prob_label} Percentile Space",
            fancybox=True,
        )
    else:
        legend1 = ax1.legend(
            *scatter.legend_elements(num=numcols),
            loc=lloc,
            title=f"{compare_element}\nRank",
            fancybox=True,
        )

    plt.setp(legend1.get_title(), multialignment="center", color=text_color)

    for text in legend1.get_texts():
        text.set_color(text_color)

    ax1.add_artist(legend1)

    if cwa_outline:
        try:
            if not os.path.exists("shp/w_22mr22.shp"):
                cwa_url = "https://www.weather.gov/source/gis/Shapefiles/WSOM/w_22mr22.zip"

                if not os.path.exists("shp"):
                    os.mkdir("shp")

                urlretrieve(cwa_url, "shp/nws_cwa_outlines.zip")

                with zipfile.ZipFile("shp/nws_cwa_outlines.zip", "r") as zip_ref:
                    zip_ref.extractall("shp")

            cwa_feature = ShapelyFeature(
                Reader("shp/w_22mr22.shp").geometries(),
                ccrs.PlateCarree(),
                edgecolor="black",
                facecolor="none",
                linewidth=1,
                linestyle="-",
                zorder=4,
            )
            ax1.add_feature(cwa_feature)

        except Exception:
            print("Aw shucks, no CWA boundaries for you. Sorry bout that.")

    if county_outline:
        try:
            if not os.path.exists("shp/c_08mr23.shp"):
                county_url = "https://www.weather.gov/source/gis/Shapefiles/County/c_08mr23.zip"

                if not os.path.exists("shp"):
                    os.mkdir("shp")

                if not os.path.exists("shp/counties.zip"):
                    urlretrieve(county_url, "shp/counties.zip")
                    print("   >> Downloaded county zip file")

                with zipfile.ZipFile("shp/counties.zip", "r") as cty_ref:
                    cty_ref.extractall("shp")
                    print("   >> Extracted county shape files")

            cty_feature = ShapelyFeature(
                Reader("shp/c_08mr23.shp").geometries(),
                ccrs.PlateCarree(),
                edgecolor="grey",
                facecolor="none",
                linewidth=0.5,
                linestyle=":",
                zorder=3,
            )
            ax1.add_feature(cty_feature)

        except Exception:
            print("   >> Cannot plot county boundaries.")

    ax2.set_anchor("C")

    sns.histplot(
        data=obs[dataframeid],
        x=compare_var,
        ax=ax2,
        kde=True,
        bins=range(-10, 115, 10),
        color="steelblue",
        edgecolor="lightgrey",
    )

    ax2.set_xlabel(
        f"{compare_element} in NBM {prob_label} Percentile Bins",
        color=text_color,
        fontsize=12,
    )

    ax2.axvline(mean, color="salmon", linestyle="--", label="Mean")
    ax2.axvline(median, color="mediumaquamarine", linestyle="-", label="Median")

    ax2.grid(False)

    for tick in ax2.get_xticklabels():
        tick.set_color(text_color)

    for tick in ax2.get_yticklabels():
        tick.set_color(text_color)

    ax2.tick_params(axis="y", labelsize=8, color=text_color)

    legend2 = ax2.legend()

    for text in legend2.get_texts():
        text.set_color(text_color)

    ax2.set(ylabel=None)


# --------------------------------------------------------------------------------------------------
# Save plot
# --------------------------------------------------------------------------------------------------

safe_nbm_version = str(NBM_VERSION_LABEL).replace(".", "p")

if custom_area:
    figname = (
        f"{custom_area_name}_map_NBMv{safe_nbm_version}_"
        f"{dataframeid}_{compare_element}_{element}_"
        f"{nbm_init.strftime('%Y%m%d_%H')}_"
        f"{fig_valid_date}_"
        f"{forecast_hour_label}.png"
    )
else:
    figname = (
        f"map_NBMv{safe_nbm_version}_"
        f"{dataframeid}_{compare_element}_{element}_"
        f"{nbm_init.strftime('%Y%m%d_%H')}_"
        f"{fig_valid_date}_"
        f"{forecast_hour_label}.png"
    )

# Clean up filename if forecast_hour_label is empty for non-valid-time products.
figname = figname.replace("__", "_").replace("_.png", ".png")

plt.savefig(
    figname,
    facecolor=fig.get_facecolor(),
    bbox_inches="tight",
    pad_inches=0.2,
    dpi="figure",
)

print(f"   > Done! Saved plot as {figname}")

In [ ]:
#@title 7. Zip up all images on the left for download

!zip -r NBM_Data.zip *.png

In [ ]:
#@title 8. Remove images
!rm *.png
!rm *.zip